# DiffuGPT-S: Discovering linguistic-relation attention heads (structured grammar)

Finds which attention heads encode a linguistic relation `attender -> receiver` (e.g. *object -> verb*) using the controlled-grammar sentences in `train_data_large.txt` / `eval_data_large.txt`.

Every sentence follows the fixed template
`DET ADJ NOUN(subj) VERB DET ADJ NOUN(obj) [adverbial PP]`, so the ground-truth role of each word is known **by position** - no LLM annotation or string matching needed.

Pipeline:
1. Parse each sentence's roles by word index and map them to token spans.
2. Run the model on each **fully unmasked** sentence; for every head take the final-step attention `[S,S]` and predict the **receiver** as the token the **attender** row attends to most (`attn.argmax(dim=1)`), masking the BOS sink and the attender's own positions.
3. For each `(layer, head, relation)` compute prediction accuracy on **train (1000 sentences)** to rank candidate heads, then **confirm on the held-out eval set (200 sentences)**.
4. Visualize each relation's top head evolving over diffusion time, with the `attender -> receiver` edge boxed, so the fully unmasked frame demonstrates the relation.

Figures are saved as Type 1 / embedded-font PDFs (`plt.rcParams['pdf.fonttype'] = 42`). Runs on a T4.

## Part 0 - Environment setup and model loading (from Raghu's notebook)

In [ ]:
!nvidia-smi

!pip install -q transformers==4.44.2 huggingface_hub

!rm -rf DiffuLLaMA && git clone --depth 1 https://github.com/HKUNLP/DiffuLLaMA.git
%cd DiffuLLaMA

import torch
from transformers import AutoConfig, AutoTokenizer
from model import DiscreteDiffusionModel, generate_samples

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL = "gpt2"  # only the config is read from this; weights come from MODEL_NAME

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME,
    model=BASE_MODEL,
    config=config,
    tokenizer=tokenizer,
    device="cuda",
).to("cuda")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}: {n_params/1e6:.1f}M params, hidden={config.hidden_size}, layers={config.num_hidden_layers}")


## Shared helpers (from Raghu's notebook)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42  # Embed fonts in PDF outputs for reproducible/publication-friendly figures.
plt.rcParams["ps.fonttype"] = 42
import pandas as pd
from model import get_anneal_attn_mask

@torch.no_grad()
def forward_with_attentions(model, input_ids, attention_mask):
    """
    Runs one DiffuGPT denoising forward pass and returns:
    - logits
    - attentions from every layer

    attentions[layer] shape:
        [batch, num_heads, seq_len, seq_len]
    """
    x_embed = model.get_embeds(input_ids)

    outputs = model.denoise_model(
        inputs_embeds=x_embed,
        attention_mask=attention_mask,
        output_attentions=True,
        output_hidden_states=False,
        return_dict=True,
        use_cache=False,
    )

    logits = model.get_logits(outputs.last_hidden_state)
    attentions = outputs.attentions

    return logits, attentions

text_sequence = "Today is a wonderful day,"

def tokenize_for_experiment(tokenizer, text_sequence, add_bos=True):
    token_ids = tokenizer.encode(text_sequence)

    if add_bos:
        token_ids = [tokenizer.bos_token_id] + token_ids

    input_ids = torch.tensor([token_ids], device=model.device)

    readable_tokens = []
    for tok_id in token_ids:
        readable_tokens.append(tokenizer.decode([tok_id]))

    return input_ids, readable_tokens

true_input_ids, readable_tokens = tokenize_for_experiment(tokenizer, text_sequence)

print("seq_len:", true_input_ids.shape[1])
for i, tok in enumerate(readable_tokens):
    print(i, repr(tok))



In [ ]:
@torch.no_grad()
def get_xt_at_diffusion_time(
    model,
    tokenizer,
    text_sequence,
    diffusion_time,
    diffusion_steps=64,
    seed=42,
    include_bos=True,
    force_final_fully_unmasked=True,
):
    """
    Returns the token state xt at a chosen diffusion time.

    diffusion_time:
        0  = beginning, mostly masked
        63 = final step, fully unmasked if force_final_fully_unmasked=True

    Returns:
        xt: token IDs at this diffusion time
        readable_tokens: token labels
        is_visible: boolean list, True if token is currently unmasked/visible
        unmask_step: step when each token became visible
    """

    assert 0 <= diffusion_time < diffusion_steps

    torch.manual_seed(seed)
    np.random.seed(seed)

    true_input_ids, readable_tokens = tokenize_for_experiment(
        tokenizer,
        text_sequence,
        add_bos=include_bos,
    )

    true_input_ids = true_input_ids.to(model.device)
    batch_size, seq_len = true_input_ids.shape

    if tokenizer.mask_token_id is None:
        raise ValueError("tokenizer.mask_token_id is None.")

    # Start from the true sentence
    xt = true_input_ids.clone()

    # Mask every token except BOS
    maskable_mask = torch.ones_like(xt, dtype=torch.bool)

    if include_bos:
        maskable_mask[:, 0] = False

    xt = xt.masked_fill(maskable_mask, tokenizer.mask_token_id)

    remaining_mask = maskable_mask.clone()

    unmask_step = [-1] * seq_len

    if include_bos:
        unmask_step[0] = 0

    # Reproduce teacher-forced random unmasking schedule
    for progress_step in range(diffusion_time):
        diffusion_t = diffusion_steps - progress_step
        p_to_unmask = 1.0 / diffusion_t

        reveal_now = remaining_mask & (
            torch.rand_like(
                remaining_mask,
                dtype=torch.float,
                device=model.device,
            ) < p_to_unmask
        )

        xt = xt.clone()
        xt[reveal_now] = true_input_ids[reveal_now]

        reveal_positions = reveal_now[0].nonzero(as_tuple=True)[0].tolist()

        for pos in reveal_positions:
            if unmask_step[pos] == -1:
                unmask_step[pos] = progress_step + 1

        remaining_mask = remaining_mask & (~reveal_now)

    # For final heatmaps, force the sentence to be fully unmasked
    if force_final_fully_unmasked and diffusion_time == diffusion_steps - 1:
        xt = true_input_ids.clone()
        remaining_mask = torch.zeros_like(remaining_mask, dtype=torch.bool)

        for pos in range(seq_len):
            if unmask_step[pos] == -1:
                unmask_step[pos] = diffusion_time

    is_visible = (~remaining_mask[0]).detach().cpu().tolist()

    return xt, readable_tokens, is_visible, unmask_step



In [ ]:
@torch.no_grad()
def get_all_attention_matrices_at_time(
    model,
    tokenizer,
    text_sequence,
    diffusion_time,
    diffusion_steps=64,
    seed=42,
    include_bos=True,

):
    """
    Gets all attention matrices at a specific diffusion time.

    Returns:
        attentions:
            tuple of length num_layers
            attentions[layer].shape = [batch, num_heads, seq_len, seq_len]

        readable_tokens:
            token labels

        is_visible:
            which tokens are visible/unmasked at this diffusion time

        unmask_step:
            when each token became visible
    """

    xt, readable_tokens, is_visible, unmask_step = get_xt_at_diffusion_time(
        model=model,
        tokenizer=tokenizer,
        text_sequence=text_sequence,
        diffusion_time=diffusion_time,
        diffusion_steps=diffusion_steps,
        seed=seed,
        include_bos=include_bos,
        force_final_fully_unmasked=True,
    )

    batch_size, seq_len = xt.shape
    x_embed = model.get_embeds(xt)

    attention_mask = get_anneal_attn_mask(
        seq_len=seq_len,
        bsz=batch_size,
        dtype=x_embed.dtype,
        device=xt.device,
        attn_mask_ratio=1.0,
    )

    logits, attentions = forward_with_attentions(
        model=model,
        input_ids=xt,
        attention_mask=attention_mask,
    )

    return attentions, readable_tokens, is_visible, unmask_step



In [ ]:
@torch.no_grad()
def collect_attention_entropy_over_time(
    model,
    tokenizer,
    text_sequence,
    layer_idx=0,
    head_idx=0,
    diffusion_steps=64,
    seed=42,
    include_bos=True,
    normalize_entropy=False,
):
    """
    Measures attention entropy over diffusion time.

    Args:
        text_sequence:
            The sentence you want to analyze.

        layer_idx:
            Which transformer layer to inspect.
            DiffuGPT-S has 12 layers, so use 0 through 11.

        head_idx:
            Which attention head to inspect.

        diffusion_steps:
            Usually 64, matching your notebook.

        seed:
            Controls the random unmasking schedule.

        normalize_entropy:
            If False, entropy is in natural-log units.
            If True, entropy is divided by log(seq_len), so values are 0 to 1.

    Returns:
        entropy_matrix:
            shape [diffusion_steps, seq_len]

        readable_tokens:
            list of token strings

        unmask_step:
            list where unmask_step[i] is the diffusion progress step when token i became visible.
            BOS is visible from the start.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    # -----------------------------
    # Tokenize
    # -----------------------------
    true_input_ids, readable_tokens = tokenize_for_experiment(
        tokenizer,
        text_sequence,
        add_bos=include_bos,
    )

    true_input_ids = true_input_ids.to(model.device)
    batch_size, seq_len = true_input_ids.shape

    # -----------------------------
    # Start with all tokens masked except BOS
    # -----------------------------
    xt = true_input_ids.clone()

    if tokenizer.mask_token_id is None:
        raise ValueError("tokenizer.mask_token_id is None. DiffuGPT should have a mask token.")

    maskable_mask = torch.ones_like(xt, dtype=torch.bool)

    if include_bos:
        maskable_mask[:, 0] = False

    xt = xt.masked_fill(maskable_mask, tokenizer.mask_token_id)

    # Track when each token becomes unmasked
    unmask_step = [-1] * seq_len
    if include_bos:
        unmask_step[0] = 0

    # -----------------------------
    # Build DiffuGPT 4D attention mask
    # -----------------------------
    x_embed = model.get_embeds(xt)

    attention_mask = get_anneal_attn_mask(
        seq_len=seq_len,
        bsz=batch_size,
        dtype=x_embed.dtype,
        device=xt.device,
        attn_mask_ratio=1.0,
    )

    # -----------------------------
    # Storage
    # -----------------------------
    all_attention_mats = []
    all_xt_states = []

    # progress_step goes 0 -> diffusion_steps - 1
    # progress_step = 0 means most masked
    # progress_step = 63 means late denoising
    remaining_mask = maskable_mask.clone()

    for progress_step in range(diffusion_steps):
        diffusion_t = diffusion_steps - progress_step

        logits, attentions = forward_with_attentions(
            model=model,
            input_ids=xt,
            attention_mask=attention_mask,
        )

        if attentions is None:
            raise RuntimeError(
                "No attentions were returned. Make sure output_attentions=True is being passed."
            )

        # attentions[layer_idx]: [batch, heads, seq, seq]
        attn = attentions[layer_idx][0, head_idx].detach().float().cpu()

        # Store [seq_len, seq_len]
        all_attention_mats.append(attn)
        all_xt_states.append(xt.detach().cpu().clone())

        # Stop after final measurement
        if progress_step == diffusion_steps - 1:
            break

        # -----------------------------
        # Reveal some still-masked tokens.
        # This mirrors the DiffuGPT random denoising schedule.
        # -----------------------------
        p_to_unmask = 1.0 / diffusion_t

        reveal_now = remaining_mask & (
            torch.rand_like(remaining_mask, dtype=torch.float, device=model.device) < p_to_unmask
        )

        xt = xt.clone()
        xt[reveal_now] = true_input_ids[reveal_now]

        # Record the progress step when each token became visible
        reveal_positions = reveal_now[0].nonzero(as_tuple=True)[0].tolist()
        for pos in reveal_positions:
            if unmask_step[pos] == -1:
                unmask_step[pos] = progress_step + 1

        remaining_mask = remaining_mask & (~reveal_now)

    # -----------------------------
    # Convert attention to entropy
    # -----------------------------
    attention_over_time = torch.stack(all_attention_mats, dim=0)
    # shape: [diffusion_steps, seq_len, seq_len]

    probs = attention_over_time.clamp_min(1e-12)
    entropy_matrix = -(probs * probs.log()).sum(dim=-1)
    # shape: [diffusion_steps, seq_len]

    if normalize_entropy:
        entropy_matrix = entropy_matrix / np.log(seq_len)

    entropy_matrix = entropy_matrix.numpy()

    return entropy_matrix, readable_tokens, unmask_step, attention_over_time.numpy(), all_xt_states


## Embedded data (no upload needed)

The controlled-grammar sentences are embedded directly below, so the notebook runs end-to-end with no file uploads. Every sentence follows `DET ADJ NOUN(subj) VERB DET ADJ NOUN(obj) [PP]`.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

DIFFUSION_STEPS = 64
FINAL_TIME = DIFFUSION_STEPS - 1     # fully unmasked frame
SEED = 42
EXCLUDE_BOS = True                   # mask the BOS attention-sink column before argmax
EXCLUDE_SELF = True                  # mask the attender's own token positions
ATTENDER_TOKEN = "last"              # which subword of a word is the attender row ("last" or "first")
MAX_TRAIN = None                     # cap #train sentences for a quick pass (None = all)
MAX_EVAL = None
OUT_DIR = "relation_head_search"
os.makedirs(OUT_DIR, exist_ok=True)

# Sentences are embedded directly so the notebook runs with no uploads.
# Controlled grammar: DET ADJ NOUN(subj) VERB DET ADJ NOUN(obj) [adverbial PP].

TRAIN_SENTENCES = [
    "The mechanical clock constructed the temporal anomaly at the speed of light.",
    "A mysterious artifact navigated the neural network with remarkable efficiency.",
    "A clever hacker discovered the magnetic field in absolute silence.",
    "The forgotten king investigated a mechanical flaw through sheer willpower.",
    "The mechanical clock shattered the radioactive isotope without hesitation.",
    "The majestic eagle observed a celestial body in complete defiance.",
    "The distant supernova overcame a psychological barrier before the deadline.",
    "A solitary lighthouse shattered a mechanical flaw before the deadline.",
    "The seasoned astronaut measured the temporal anomaly in a spectacular display.",
    "A fierce gladiator calculated the fundamental theorem without any prior warning.",
    "The gentle healer calibrated a psychological barrier in the blink of an eye.",
    "A quantum computer overcame a delicate ecosystem without leaving a trace.",
    "The weary traveler measured a microscopic organism through sheer willpower.",
    "The sprawling city analyzed the kinetic energy without hesitation.",
    "The sprawling city bypassed a philosophical concept with subtle elegance.",
    "A quantum computer discovered the kinetic energy in absolute silence.",
    "The mechanical clock challenged a digital footprint for future generations.",
    "The ancient philosopher challenged the kinetic energy with devastating consequences.",
    "The sprawling city overcame a fading memory with mathematical certainty.",
    "The massive whale challenged an intricate puzzle for the sake of progress.",
    "A clever hacker discovered the ancient ruins in the blink of an eye.",
    "A solitary lighthouse overcame a hidden frequency with devastating consequences.",
    "A brilliant detective transformed the encrypted data with remarkable efficiency.",
    "An autonomous robot overcame a fading memory in complete defiance.",
    "A wandering musician discovered a digital footprint at the speed of light.",
    "A glowing embers observed a digital footprint in complete defiance.",
    "A fierce gladiator manipulated a hidden frequency at the speed of light.",
    "The mechanical clock challenged a psychological barrier in absolute silence.",
    "A fierce gladiator discovered the quantum state with cautious optimism.",
    "A solitary lighthouse investigated the historical archive with reckless abandon.",
    "The majestic eagle overcame the structural integrity during the final phase.",
    "The weary traveler synthesized the atmospheric pressure using unprecedented methods.",
    "The relentless storm protected the magnetic field with mathematical certainty.",
    "A brilliant detective generated the kinetic energy without any prior warning.",
    "The silent observer observed a distant galaxy through rigorous testing.",
    "A fierce gladiator amplified a mechanical flaw without leaving a trace.",
    "The weary traveler calibrated a delicate ecosystem with reckless abandon.",
    "A quantum computer repaired the dynamic equilibrium under the cover of darkness.",
    "The majestic eagle measured a virtual environment against all odds.",
    "The majestic eagle illuminated the atmospheric pressure before the deadline.",
    "The distant supernova challenged a fading memory using advanced techniques.",
    "The ancient philosopher amplified the dynamic equilibrium in perfect harmony.",
    "The sprawling city challenged the atmospheric pressure in absolute silence.",
    "A solitary lighthouse protected the fundamental theorem beyond expected parameters.",
    "A quantum computer bypassed a virtual environment for future generations.",
    "The swift fox orchestrated the temporal anomaly without leaving a trace.",
    "A hidden waterfall shattered a delicate ecosystem in absolute silence.",
    "A fragile butterfly calculated a classified document without leaving a trace.",
    "The weary traveler highlighted the fundamental theorem without hesitation.",
    "A mysterious artifact amplified the radioactive isotope beyond expected parameters.",
    "A curious child shattered a digital footprint for future generations.",
    "The massive whale manipulated a philosophical concept in complete defiance.",
    "A glowing embers manipulated a delicate ecosystem through sheer willpower.",
    "The majestic eagle calculated the harmonic resonance in the blink of an eye.",
    "The forgotten king constructed a rogue signal at the speed of light.",
    "The loyal companion embraced a celestial body in perfect harmony.",
    "An elegant dancer calculated a celestial body for future generations.",
    "The mechanical clock calculated the harmonic resonance with mathematical certainty.",
    "The sprawling city protected an intricate puzzle for future generations.",
    "A mysterious artifact illuminated the ancient ruins using unprecedented methods.",
    "A brilliant detective decoded the harmonic resonance with subtle elegance.",
    "The loyal companion transformed a classified document without hesitation.",
    "The massive whale bypassed the radioactive isotope in a spectacular display.",
    "The gentle healer neutralized the structural integrity in a controlled environment.",
    "The silent observer orchestrated the magnetic field using advanced techniques.",
    "A quantum computer shattered a fading memory during the final phase.",
    "The mechanical clock observed the encrypted data using unprecedented methods.",
    "The loyal companion amplified the structural integrity through rigorous testing.",
    "The seasoned astronaut bypassed a virtual environment despite the inherent risks.",
    "The loyal companion manipulated a distant galaxy in the blink of an eye.",
    "The massive whale calibrated the atmospheric pressure for the sake of progress.",
    "The mechanical clock evaluated the radioactive isotope without leaving a trace.",
    "A hidden waterfall shattered a hidden frequency during the final phase.",
    "A fierce gladiator illuminated the complex algorithm through rigorous testing.",
    "A solitary lighthouse bypassed the neural network using a novel approach.",
    "The massive whale transformed a fading memory for the sake of progress.",
    "The relentless storm embraced the neural network in a spectacular display.",
    "The silent observer constructed the fundamental theorem with astonishing precision.",
    "The loyal companion measured a virtual environment at the speed of light.",
    "An elegant dancer constructed the radioactive isotope with reckless abandon.",
    "An eager student discovered the historical archive with subtle elegance.",
    "A fragile butterfly amplified the temporal anomaly at the speed of light.",
    "The sprawling city calculated a classified document with devastating consequences.",
    "The sprawling city calculated a microscopic organism using a novel approach.",
    "An elegant dancer transformed a delicate ecosystem without leaving a trace.",
    "The loyal companion neutralized the temporal anomaly despite the inherent risks.",
    "The mechanical clock discovered the encrypted data in a controlled environment.",
    "The weary traveler discovered the temporal anomaly in the blink of an eye.",
    "A glowing embers discovered a mechanical flaw without leaving a trace.",
    "A solitary lighthouse manipulated the encrypted data before the deadline.",
    "The persistent researcher illuminated the structural integrity without leaving a trace.",
    "The sprawling city shattered a classified document for the sake of progress.",
    "The persistent researcher navigated the fundamental theorem with devastating consequences.",
    "The massive whale measured a digital footprint against all odds.",
    "A fragile butterfly synthesized a digital footprint without any prior warning.",
    "The sprawling city analyzed a fading memory with mathematical certainty.",
    "The mechanical clock calculated the radioactive isotope in perfect harmony.",
    "The majestic eagle repaired the harmonic resonance using advanced techniques.",
    "The gentle healer simulated a celestial body through rigorous testing.",
    "The forgotten king overcame the temporal anomaly during the final phase.",
    "The forgotten king generated a mechanical flaw with subtle elegance.",
    "The mechanical clock analyzed the neural network before the deadline.",
    "An autonomous robot calibrated the historical archive in absolute silence.",
    "The distant supernova highlighted a classified document through rigorous testing.",
    "A curious child dismantled a logical paradox with astonishing precision.",
    "The persistent researcher navigated a distant galaxy without hesitation.",
    "The ancient philosopher generated a classified document in a controlled environment.",
    "A clever hacker discovered a celestial body for future generations.",
    "The swift fox embraced a fading memory for future generations.",
    "The mechanical clock illuminated the quantum state with cautious optimism.",
    "The weary traveler orchestrated the ancient ruins without hesitation.",
    "A curious child neutralized an intricate puzzle despite the inherent risks.",
    "An autonomous robot discovered the fundamental theorem without leaving a trace.",
    "A solitary lighthouse decoded the temporal anomaly using advanced techniques.",
    "A curious child analyzed a distant galaxy with subtle elegance.",
    "A quantum computer repaired the complex algorithm before the deadline.",
    "The weary traveler investigated the complex algorithm with mathematical certainty.",
    "An elegant dancer observed the encrypted data without leaving a trace.",
    "A mysterious artifact analyzed the ancient ruins without any prior warning.",
    "The seasoned astronaut highlighted the dynamic equilibrium in a spectacular display.",
    "A glowing embers challenged a microscopic organism despite the inherent risks.",
    "An autonomous robot amplified the temporal anomaly despite the inherent risks.",
    "The sprawling city manipulated the quantum state for the sake of progress.",
    "The weary traveler abandoned a distant galaxy through rigorous testing.",
    "The silent observer constructed a mechanical flaw at the speed of light.",
    "The relentless storm synthesized a hidden frequency during the final phase.",
    "The massive whale overcame a rogue signal in complete defiance.",
    "A mysterious artifact observed a philosophical concept at the speed of light.",
    "The silent observer calibrated a psychological barrier despite the inherent risks.",
    "The gentle healer synthesized a hidden frequency for future generations.",
    "The loyal companion bypassed a mechanical flaw with subtle elegance.",
    "An autonomous robot analyzed a rogue signal using a novel approach.",
    "The persistent researcher measured an intricate puzzle in absolute silence.",
    "The silent observer constructed the structural integrity in the blink of an eye.",
    "A mysterious artifact orchestrated a delicate ecosystem under the cover of darkness.",
    "The silent observer investigated the fundamental theorem with astonishing precision.",
    "A wandering musician shattered a logical paradox at the speed of light.",
    "A glowing embers embraced the temporal anomaly through rigorous testing.",
    "The persistent researcher neutralized the fundamental theorem with mathematical certainty.",
    "A fierce gladiator transformed a classified document beyond expected parameters.",
    "A fierce gladiator neutralized an intricate puzzle despite the inherent risks.",
    "A mysterious artifact discovered the radioactive isotope with subtle elegance.",
    "A glowing embers discovered the magnetic field in a controlled environment.",
    "A fragile butterfly challenged the harmonic resonance without hesitation.",
    "A rogue asteroid discovered a hidden frequency with cautious optimism.",
    "The seasoned astronaut investigated the structural integrity using advanced techniques.",
    "The loyal companion observed the harmonic resonance with reckless abandon.",
    "The loyal companion decoded the temporal anomaly in the blink of an eye.",
    "The loyal companion dismantled the encrypted data in complete defiance.",
    "The distant supernova dismantled a psychological barrier without leaving a trace.",
    "The distant supernova generated a mechanical flaw against all odds.",
    "The massive whale orchestrated a distant galaxy despite the inherent risks.",
    "A fragile butterfly neutralized the magnetic field with remarkable efficiency.",
    "A brilliant detective abandoned a logical paradox through sheer willpower.",
    "The forgotten king decoded the temporal anomaly in the blink of an eye.",
    "The massive whale observed a celestial body during the final phase.",
    "The massive whale overcame the quantum state with mathematical certainty.",
    "A wandering musician evaluated the magnetic field using unprecedented methods.",
    "The silent observer transformed a rogue signal through sheer willpower.",
    "A quantum computer discovered the kinetic energy before the deadline.",
    "A quantum computer discovered a rogue signal using a novel approach.",
    "The seasoned astronaut discovered the kinetic energy before the deadline.",
    "The forgotten king illuminated the historical archive without any prior warning.",
    "The weary traveler evaluated a fading memory in perfect harmony.",
    "An autonomous robot evaluated the encrypted data with subtle elegance.",
    "The seasoned astronaut investigated the neural network through rigorous testing.",
    "A solitary lighthouse synthesized a psychological barrier for the sake of progress.",
    "A hidden waterfall abandoned a mechanical flaw for the sake of progress.",
    "A fierce gladiator shattered a classified document at the speed of light.",
    "A brilliant detective repaired a fading memory in complete defiance.",
    "The sprawling city navigated the fundamental theorem for future generations.",
    "A rogue asteroid investigated a microscopic organism without hesitation.",
    "A fierce gladiator analyzed a celestial body under the cover of darkness.",
    "The mechanical clock amplified a logical paradox in perfect harmony.",
    "An autonomous robot investigated a hidden frequency in complete defiance.",
    "An elegant dancer investigated an intricate puzzle with subtle elegance.",
    "A rogue asteroid manipulated a distant galaxy beyond expected parameters.",
    "The forgotten king simulated a celestial body for future generations.",
    "The gentle healer highlighted the atmospheric pressure with remarkable efficiency.",
    "The seasoned astronaut synthesized the dynamic equilibrium for the sake of progress.",
    "A curious child simulated the fundamental theorem without leaving a trace.",
    "A wandering musician navigated a psychological barrier through sheer willpower.",
    "A fragile butterfly navigated a celestial body using unprecedented methods.",
    "The mechanical clock constructed the fundamental theorem through rigorous testing.",
    "The majestic eagle constructed a delicate ecosystem in complete defiance.",
    "A mysterious artifact calibrated the quantum state in a controlled environment.",
    "The forgotten king evaluated the quantum state without leaving a trace.",
    "A mysterious artifact highlighted a distant galaxy without leaving a trace.",
    "A fierce gladiator challenged the neural network in perfect harmony.",
    "The ancient philosopher measured a hidden frequency using a novel approach.",
    "An eager student amplified the structural integrity with reckless abandon.",
    "An eager student constructed the neural network through sheer willpower.",
    "The persistent researcher embraced the neural network for future generations.",
    "An elegant dancer highlighted the radioactive isotope using unprecedented methods.",
    "The forgotten king transformed the harmonic resonance during the final phase.",
    "The sprawling city calculated the fundamental theorem using advanced techniques.",
    "An elegant dancer generated a virtual environment in complete defiance.",
    "A brilliant detective investigated a virtual environment with reckless abandon.",
    "An eager student bypassed the complex algorithm with subtle elegance.",
    "An autonomous robot discovered the kinetic energy with remarkable efficiency.",
    "A brilliant detective embraced the ancient ruins at the speed of light.",
    "The relentless storm calibrated a hidden frequency with remarkable efficiency.",
    "The mechanical clock measured a philosophical concept with devastating consequences.",
    "The massive whale calibrated a virtual environment at the speed of light.",
    "The relentless storm analyzed the atmospheric pressure in a spectacular display.",
    "A clever hacker protected a celestial body before the deadline.",
    "A brilliant detective orchestrated a fading memory with reckless abandon.",
    "The silent observer bypassed a philosophical concept using unprecedented methods.",
    "The sprawling city evaluated a celestial body with subtle elegance.",
    "The distant supernova generated a philosophical concept with reckless abandon.",
    "The sprawling city manipulated a distant galaxy without hesitation.",
    "The weary traveler manipulated a mechanical flaw with reckless abandon.",
    "A clever hacker decoded a mechanical flaw for the sake of progress.",
    "A wandering musician measured the complex algorithm in complete defiance.",
    "The swift fox orchestrated a celestial body in perfect harmony.",
    "The seasoned astronaut amplified the radioactive isotope using a novel approach.",
    "A brilliant detective investigated the complex algorithm despite the inherent risks.",
    "A fierce gladiator calculated the radioactive isotope with mathematical certainty.",
    "A clever hacker dismantled the fundamental theorem in complete defiance.",
    "The seasoned astronaut challenged a fading memory using unprecedented methods.",
    "A fragile butterfly synthesized an intricate puzzle in a spectacular display.",
    "A rogue asteroid evaluated a logical paradox using unprecedented methods.",
    "A curious child simulated a digital footprint in a spectacular display.",
    "The sprawling city analyzed the atmospheric pressure through rigorous testing.",
    "The loyal companion challenged a hidden frequency with cautious optimism.",
    "The ancient philosopher investigated a virtual environment for the sake of progress.",
    "The mechanical clock amplified the ancient ruins in complete defiance.",
    "A mysterious artifact simulated a hidden frequency using advanced techniques.",
    "A hidden waterfall evaluated a psychological barrier through sheer willpower.",
    "The swift fox neutralized a microscopic organism under the cover of darkness.",
    "The forgotten king challenged the quantum state without any prior warning.",
    "A brilliant detective analyzed a celestial body against all odds.",
    "The swift fox simulated the neural network with reckless abandon.",
    "A solitary lighthouse observed the historical archive with devastating consequences.",
    "A quantum computer simulated the atmospheric pressure with subtle elegance.",
    "A mysterious artifact simulated the magnetic field against all odds.",
    "The massive whale orchestrated the harmonic resonance with cautious optimism.",
    "A clever hacker protected the fundamental theorem using unprecedented methods.",
    "The seasoned astronaut overcame the harmonic resonance in the blink of an eye.",
    "The mechanical clock neutralized a virtual environment in complete defiance.",
    "The seasoned astronaut observed a logical paradox using advanced techniques.",
    "A wandering musician abandoned the neural network despite the inherent risks.",
    "A fierce gladiator repaired the magnetic field despite the inherent risks.",
    "The majestic eagle measured a mechanical flaw with subtle elegance.",
    "The ancient philosopher illuminated the magnetic field for future generations.",
    "A wandering musician constructed a digital footprint during the final phase.",
    "A fierce gladiator observed the fundamental theorem without any prior warning.",
    "The gentle healer navigated the atmospheric pressure using unprecedented methods.",
    "The distant supernova embraced a classified document for the sake of progress.",
    "The persistent researcher measured a classified document through rigorous testing.",
    "A glowing embers calibrated the atmospheric pressure without leaving a trace.",
    "The persistent researcher calculated a delicate ecosystem using advanced techniques.",
    "The silent observer neutralized the atmospheric pressure in perfect harmony.",
    "A glowing embers analyzed the radioactive isotope without hesitation.",
    "An eager student measured the ancient ruins in complete defiance.",
    "A fierce gladiator evaluated the dynamic equilibrium in the blink of an eye.",
    "The forgotten king generated a logical paradox for future generations.",
    "A rogue asteroid observed the fundamental theorem using advanced techniques.",
    "A rogue asteroid highlighted a fading memory in absolute silence.",
    "The forgotten king evaluated a virtual environment using unprecedented methods.",
    "The majestic eagle bypassed the structural integrity during the final phase.",
    "The distant supernova calibrated the complex algorithm before the deadline.",
    "A fragile butterfly evaluated the atmospheric pressure against all odds.",
    "The ancient philosopher neutralized a logical paradox against all odds.",
    "An eager student orchestrated a virtual environment through sheer willpower.",
    "The gentle healer illuminated the ancient ruins through sheer willpower.",
    "A brilliant detective manipulated the fundamental theorem with subtle elegance.",
    "A curious child measured the harmonic resonance before the deadline.",
    "The weary traveler generated a classified document in absolute silence.",
    "A fierce gladiator measured a delicate ecosystem without leaving a trace.",
    "The majestic eagle discovered a virtual environment in complete defiance.",
    "A solitary lighthouse protected a psychological barrier without any prior warning.",
    "The relentless storm observed the neural network during the final phase.",
    "The sprawling city neutralized the radioactive isotope with mathematical certainty.",
    "A wandering musician neutralized the complex algorithm in absolute silence.",
    "The massive whale observed the magnetic field in complete defiance.",
    "The weary traveler discovered the magnetic field with astonishing precision.",
    "A glowing embers analyzed a celestial body in a spectacular display.",
    "The swift fox challenged the atmospheric pressure in complete defiance.",
    "The massive whale illuminated a logical paradox in a spectacular display.",
    "The mechanical clock abandoned the magnetic field using a novel approach.",
    "The relentless storm generated a logical paradox with astonishing precision.",
    "The gentle healer transformed the magnetic field in perfect harmony.",
    "A brilliant detective calibrated the complex algorithm through rigorous testing.",
    "A wandering musician investigated the ancient ruins using advanced techniques.",
    "The silent observer synthesized a mechanical flaw despite the inherent risks.",
    "A rogue asteroid simulated the encrypted data in complete defiance.",
    "A clever hacker measured the kinetic energy with astonishing precision.",
    "The relentless storm amplified the ancient ruins with cautious optimism.",
    "An autonomous robot challenged the magnetic field beyond expected parameters.",
    "A wandering musician constructed the complex algorithm through rigorous testing.",
    "The weary traveler shattered an intricate puzzle with devastating consequences.",
    "The majestic eagle analyzed the radioactive isotope using unprecedented methods.",
    "The loyal companion orchestrated the structural integrity with devastating consequences.",
    "The silent observer shattered a microscopic organism with cautious optimism.",
    "The massive whale navigated the kinetic energy through sheer willpower.",
    "A curious child calculated the ancient ruins in absolute silence.",
    "A fragile butterfly calibrated the temporal anomaly despite the inherent risks.",
    "A curious child illuminated a celestial body with reckless abandon.",
    "An elegant dancer dismantled a rogue signal with devastating consequences.",
    "The distant supernova embraced a fading memory at the speed of light.",
    "The loyal companion synthesized a delicate ecosystem beyond expected parameters.",
    "The silent observer investigated the kinetic energy without hesitation.",
    "The forgotten king manipulated the atmospheric pressure under the cover of darkness.",
    "A fierce gladiator dismantled an intricate puzzle against all odds.",
    "A clever hacker bypassed the structural integrity using advanced techniques.",
    "A curious child illuminated the complex algorithm in complete defiance.",
    "A quantum computer constructed the historical archive against all odds.",
    "A solitary lighthouse simulated the neural network in a controlled environment.",
    "A hidden waterfall calibrated the dynamic equilibrium with cautious optimism.",
    "A clever hacker repaired a philosophical concept with cautious optimism.",
    "The forgotten king generated the encrypted data beyond expected parameters.",
    "A glowing embers repaired the magnetic field for future generations.",
    "A hidden waterfall observed the complex algorithm with mathematical certainty.",
    "The sprawling city orchestrated the radioactive isotope using a novel approach.",
    "A clever hacker repaired the temporal anomaly using unprecedented methods.",
    "The majestic eagle protected the temporal anomaly in absolute silence.",
    "The massive whale bypassed a rogue signal through rigorous testing.",
    "The mechanical clock calculated an intricate puzzle with remarkable efficiency.",
    "The distant supernova calibrated the ancient ruins with devastating consequences.",
    "The loyal companion repaired the ancient ruins in a controlled environment.",
    "An autonomous robot amplified a logical paradox for future generations.",
    "The swift fox decoded the harmonic resonance in a controlled environment.",
    "An elegant dancer calculated the ancient ruins with mathematical certainty.",
    "The persistent researcher bypassed an intricate puzzle using advanced techniques.",
    "The seasoned astronaut shattered a delicate ecosystem in absolute silence.",
    "An autonomous robot generated the historical archive in complete defiance.",
    "A solitary lighthouse observed a digital footprint using unprecedented methods.",
    "A glowing embers measured the neural network using unprecedented methods.",
    "A hidden waterfall decoded the radioactive isotope with mathematical certainty.",
    "The loyal companion analyzed the encrypted data during the final phase.",
    "The weary traveler neutralized the dynamic equilibrium in a controlled environment.",
    "The forgotten king bypassed the radioactive isotope through sheer willpower.",
    "A brilliant detective orchestrated the fundamental theorem despite the inherent risks.",
    "A curious child decoded the temporal anomaly in perfect harmony.",
    "An eager student amplified a microscopic organism through sheer willpower.",
    "A wandering musician calculated an intricate puzzle with astonishing precision.",
    "A glowing embers generated the historical archive without leaving a trace.",
    "The gentle healer challenged a philosophical concept with cautious optimism.",
    "The gentle healer analyzed a microscopic organism with cautious optimism.",
    "A wandering musician generated a fading memory with cautious optimism.",
    "A mysterious artifact dismantled the quantum state despite the inherent risks.",
    "A fierce gladiator transformed the encrypted data despite the inherent risks.",
    "A solitary lighthouse decoded a rogue signal with subtle elegance.",
    "The mechanical clock protected a delicate ecosystem using unprecedented methods.",
    "A brilliant detective constructed an intricate puzzle during the final phase.",
    "A mysterious artifact generated the harmonic resonance in complete defiance.",
    "A wandering musician challenged a hidden frequency with cautious optimism.",
    "The loyal companion overcame a mechanical flaw through sheer willpower.",
    "The silent observer manipulated the temporal anomaly without hesitation.",
    "A mysterious artifact challenged the radioactive isotope against all odds.",
    "The distant supernova shattered an intricate puzzle in the blink of an eye.",
    "A fragile butterfly dismantled the neural network in complete defiance.",
    "A quantum computer measured the quantum state without any prior warning.",
    "A wandering musician evaluated a distant galaxy in a controlled environment.",
    "A brilliant detective neutralized the harmonic resonance against all odds.",
    "The sprawling city repaired a digital footprint beyond expected parameters.",
    "A mysterious artifact simulated a philosophical concept in complete defiance.",
    "A curious child generated a fading memory against all odds.",
    "The sprawling city measured an intricate puzzle under the cover of darkness.",
    "The relentless storm analyzed a philosophical concept under the cover of darkness.",
    "The gentle healer illuminated the structural integrity using unprecedented methods.",
    "A fragile butterfly evaluated the fundamental theorem through rigorous testing.",
    "An elegant dancer neutralized the harmonic resonance with reckless abandon.",
    "A wandering musician embraced a distant galaxy before the deadline.",
    "The majestic eagle repaired a celestial body with reckless abandon.",
    "The seasoned astronaut abandoned the complex algorithm using a novel approach.",
    "A mysterious artifact shattered a celestial body for the sake of progress.",
    "The relentless storm synthesized a fading memory for the sake of progress.",
    "A fragile butterfly calibrated a virtual environment through sheer willpower.",
    "An eager student challenged the dynamic equilibrium with remarkable efficiency.",
    "A brilliant detective abandoned the magnetic field with astonishing precision.",
    "A hidden waterfall overcame a fading memory despite the inherent risks.",
    "A clever hacker discovered the structural integrity with subtle elegance.",
    "The ancient philosopher simulated a logical paradox for future generations.",
    "A fierce gladiator repaired the historical archive in a spectacular display.",
    "The distant supernova dismantled a psychological barrier with remarkable efficiency.",
    "A hidden waterfall calculated the encrypted data in a controlled environment.",
    "The silent observer calibrated a mechanical flaw before the deadline.",
    "A clever hacker investigated the kinetic energy using unprecedented methods.",
    "An eager student analyzed a logical paradox in a controlled environment.",
    "The forgotten king bypassed the magnetic field without leaving a trace.",
    "An eager student decoded the radioactive isotope for future generations.",
    "The relentless storm amplified the encrypted data in perfect harmony.",
    "The loyal companion observed the harmonic resonance without hesitation.",
    "A mysterious artifact abandoned the quantum state beyond expected parameters.",
    "A fragile butterfly simulated a microscopic organism at the speed of light.",
    "A brilliant detective observed the atmospheric pressure in a spectacular display.",
    "A solitary lighthouse measured a fading memory using advanced techniques.",
    "A quantum computer highlighted a distant galaxy without leaving a trace.",
    "The silent observer protected the fundamental theorem beyond expected parameters.",
    "The swift fox calibrated the fundamental theorem through rigorous testing.",
    "A fierce gladiator decoded the complex algorithm through rigorous testing.",
    "The ancient philosopher calculated the historical archive in perfect harmony.",
    "A clever hacker neutralized the historical archive before the deadline.",
    "The silent observer repaired the complex algorithm without hesitation.",
    "An eager student neutralized an intricate puzzle during the final phase.",
    "The massive whale orchestrated the harmonic resonance through sheer willpower.",
    "The seasoned astronaut calculated a delicate ecosystem in a controlled environment.",
    "A wandering musician overcame a mechanical flaw in perfect harmony.",
    "The majestic eagle overcame the ancient ruins with cautious optimism.",
    "A mysterious artifact shattered the dynamic equilibrium against all odds.",
    "The silent observer measured the magnetic field in perfect harmony.",
    "A curious child illuminated the magnetic field in a controlled environment.",
    "The ancient philosopher orchestrated the ancient ruins at the speed of light.",
    "An eager student discovered a microscopic organism in absolute silence.",
    "A wandering musician repaired the neural network beyond expected parameters.",
    "A fierce gladiator dismantled a celestial body in the blink of an eye.",
    "The weary traveler embraced the complex algorithm for the sake of progress.",
    "A quantum computer shattered the neural network using advanced techniques.",
    "The relentless storm transformed a rogue signal for future generations.",
    "The gentle healer evaluated the complex algorithm with mathematical certainty.",
    "The relentless storm simulated a classified document at the speed of light.",
    "A brilliant detective decoded a philosophical concept through rigorous testing.",
    "A glowing embers investigated the harmonic resonance without hesitation.",
    "A glowing embers abandoned the temporal anomaly under the cover of darkness.",
    "A wandering musician simulated the complex algorithm with reckless abandon.",
    "The sprawling city calculated the radioactive isotope through rigorous testing.",
    "A hidden waterfall discovered a classified document without hesitation.",
    "A wandering musician manipulated a virtual environment in perfect harmony.",
    "A glowing embers analyzed the historical archive through rigorous testing.",
    "The swift fox evaluated a celestial body without leaving a trace.",
    "The loyal companion transformed a rogue signal in complete defiance.",
    "A fierce gladiator transformed the quantum state for future generations.",
    "A rogue asteroid orchestrated a celestial body before the deadline.",
    "The silent observer illuminated the quantum state in perfect harmony.",
    "The majestic eagle embraced the encrypted data in absolute silence.",
    "The sprawling city calibrated a philosophical concept in complete defiance.",
    "A quantum computer discovered the historical archive in a controlled environment.",
    "A wandering musician analyzed a hidden frequency in a controlled environment.",
    "The distant supernova discovered the encrypted data under the cover of darkness.",
    "A fierce gladiator simulated a distant galaxy with devastating consequences.",
    "The relentless storm protected the quantum state in a spectacular display.",
    "The weary traveler calculated a classified document for the sake of progress.",
    "A clever hacker evaluated the historical archive through sheer willpower.",
    "The mechanical clock shattered a philosophical concept at the speed of light.",
    "A fierce gladiator constructed a philosophical concept without any prior warning.",
    "The swift fox dismantled the ancient ruins during the final phase.",
    "A glowing embers embraced a delicate ecosystem during the final phase.",
    "A solitary lighthouse investigated the atmospheric pressure against all odds.",
    "A clever hacker discovered the neural network in complete defiance.",
    "A solitary lighthouse embraced a fading memory through sheer willpower.",
    "The forgotten king discovered a celestial body with astonishing precision.",
    "The sprawling city orchestrated the dynamic equilibrium with subtle elegance.",
    "A glowing embers measured a classified document using advanced techniques.",
    "A curious child synthesized a distant galaxy in a controlled environment.",
    "The weary traveler discovered a celestial body with cautious optimism.",
    "An elegant dancer illuminated a philosophical concept with remarkable efficiency.",
    "The weary traveler investigated a delicate ecosystem without leaving a trace.",
    "The ancient philosopher neutralized the dynamic equilibrium despite the inherent risks.",
    "The mechanical clock protected an intricate puzzle using advanced techniques.",
    "The weary traveler abandoned a distant galaxy without any prior warning.",
    "A hidden waterfall constructed the historical archive with astonishing precision.",
    "An elegant dancer embraced the structural integrity without leaving a trace.",
    "An elegant dancer shattered the complex algorithm at the speed of light.",
    "The gentle healer analyzed the fundamental theorem through rigorous testing.",
    "An eager student investigated the kinetic energy using a novel approach.",
    "A clever hacker neutralized the temporal anomaly without any prior warning.",
    "A brilliant detective constructed the historical archive with reckless abandon.",
    "The gentle healer dismantled the structural integrity before the deadline.",
    "The forgotten king transformed a celestial body with subtle elegance.",
    "The forgotten king constructed a virtual environment in a spectacular display.",
    "The swift fox analyzed the fundamental theorem with subtle elegance.",
    "The silent observer simulated the magnetic field before the deadline.",
    "The weary traveler embraced a classified document before the deadline.",
    "A mysterious artifact embraced the complex algorithm for the sake of progress.",
    "The loyal companion navigated the ancient ruins in the blink of an eye.",
    "A curious child repaired the historical archive in a controlled environment.",
    "A wandering musician discovered a mechanical flaw without leaving a trace.",
    "The forgotten king constructed an intricate puzzle through sheer willpower.",
    "A hidden waterfall abandoned the atmospheric pressure in absolute silence.",
    "The distant supernova simulated the structural integrity with devastating consequences.",
    "The persistent researcher investigated a hidden frequency in a controlled environment.",
    "A curious child observed the fundamental theorem despite the inherent risks.",
    "An elegant dancer synthesized the temporal anomaly with mathematical certainty.",
    "The swift fox bypassed the dynamic equilibrium without hesitation.",
    "A solitary lighthouse neutralized the neural network without hesitation.",
    "The majestic eagle protected a mechanical flaw with subtle elegance.",
    "A brilliant detective calculated the magnetic field under the cover of darkness.",
    "A glowing embers decoded a hidden frequency with reckless abandon.",
    "A curious child dismantled the ancient ruins beyond expected parameters.",
    "A quantum computer measured the complex algorithm with reckless abandon.",
    "The massive whale calculated the harmonic resonance for future generations.",
    "A hidden waterfall bypassed the radioactive isotope using a novel approach.",
    "The forgotten king generated the structural integrity before the deadline.",
    "The ancient philosopher discovered a logical paradox with mathematical certainty.",
    "The mechanical clock amplified a distant galaxy through sheer willpower.",
    "A solitary lighthouse measured the magnetic field in absolute silence.",
    "The majestic eagle amplified the quantum state in a spectacular display.",
    "A clever hacker calibrated the quantum state with subtle elegance.",
    "The swift fox shattered the neural network with remarkable efficiency.",
    "A mysterious artifact repaired the complex algorithm for the sake of progress.",
    "A brilliant detective repaired a fading memory using unprecedented methods.",
    "An eager student protected the structural integrity in the blink of an eye.",
    "The sprawling city synthesized the dynamic equilibrium in complete defiance.",
    "The forgotten king illuminated a digital footprint against all odds.",
    "The forgotten king dismantled a distant galaxy beyond expected parameters.",
    "The loyal companion calculated the fundamental theorem against all odds.",
    "A hidden waterfall generated the historical archive for the sake of progress.",
    "A glowing embers bypassed a microscopic organism with devastating consequences.",
    "The sprawling city dismantled the radioactive isotope without any prior warning.",
    "The sprawling city synthesized the encrypted data despite the inherent risks.",
    "A fierce gladiator synthesized the radioactive isotope with cautious optimism.",
    "A brilliant detective shattered the harmonic resonance under the cover of darkness.",
    "A wandering musician simulated the historical archive beyond expected parameters.",
    "A solitary lighthouse discovered a logical paradox in a spectacular display.",
    "The sprawling city discovered the neural network without hesitation.",
    "The relentless storm synthesized the atmospheric pressure with subtle elegance.",
    "A fragile butterfly analyzed a fading memory in complete defiance.",
    "A brilliant detective repaired the complex algorithm with subtle elegance.",
    "A rogue asteroid investigated the ancient ruins using a novel approach.",
    "A brilliant detective repaired a hidden frequency using a novel approach.",
    "A fierce gladiator orchestrated the complex algorithm with reckless abandon.",
    "A curious child highlighted the atmospheric pressure with cautious optimism.",
    "A glowing embers bypassed the complex algorithm without any prior warning.",
    "The sprawling city manipulated a psychological barrier with astonishing precision.",
    "An eager student calibrated the magnetic field during the final phase.",
    "A wandering musician manipulated the kinetic energy through sheer willpower.",
    "A glowing embers synthesized the historical archive with cautious optimism.",
    "A mysterious artifact neutralized a distant galaxy for the sake of progress.",
    "A brilliant detective protected the quantum state beyond expected parameters.",
    "A curious child navigated a virtual environment with cautious optimism.",
    "The loyal companion constructed a delicate ecosystem through rigorous testing.",
    "A quantum computer investigated a virtual environment with reckless abandon.",
    "A fierce gladiator illuminated a psychological barrier before the deadline.",
    "The forgotten king investigated the fundamental theorem at the speed of light.",
    "The majestic eagle synthesized an intricate puzzle without any prior warning.",
    "A wandering musician calibrated a logical paradox with reckless abandon.",
    "The ancient philosopher neutralized the harmonic resonance under the cover of darkness.",
    "The mechanical clock navigated a psychological barrier without hesitation.",
    "A fragile butterfly discovered a classified document with remarkable efficiency.",
    "A solitary lighthouse amplified the radioactive isotope without hesitation.",
    "A fierce gladiator discovered an intricate puzzle beyond expected parameters.",
    "The seasoned astronaut neutralized a delicate ecosystem in the blink of an eye.",
    "A fragile butterfly measured the quantum state against all odds.",
    "The seasoned astronaut evaluated a logical paradox without leaving a trace.",
    "The relentless storm decoded the encrypted data using a novel approach.",
    "The massive whale calculated the harmonic resonance in complete defiance.",
    "The gentle healer simulated a logical paradox without leaving a trace.",
    "The ancient philosopher transformed a distant galaxy against all odds.",
    "The swift fox bypassed the ancient ruins with remarkable efficiency.",
    "A glowing embers protected the dynamic equilibrium with subtle elegance.",
    "The sprawling city constructed the dynamic equilibrium using a novel approach.",
    "The loyal companion abandoned a hidden frequency in a controlled environment.",
    "A quantum computer amplified a psychological barrier before the deadline.",
    "The seasoned astronaut constructed the temporal anomaly with subtle elegance.",
    "The majestic eagle navigated a digital footprint before the deadline.",
    "A glowing embers navigated the complex algorithm before the deadline.",
    "A fierce gladiator evaluated a microscopic organism in a controlled environment.",
    "The silent observer bypassed a rogue signal using a novel approach.",
    "A wandering musician calibrated the temporal anomaly with reckless abandon.",
    "An eager student amplified the historical archive during the final phase.",
    "An elegant dancer overcame a rogue signal with cautious optimism.",
    "A clever hacker challenged the harmonic resonance using advanced techniques.",
    "An eager student analyzed a delicate ecosystem in the blink of an eye.",
    "A fragile butterfly orchestrated an intricate puzzle using unprecedented methods.",
    "A hidden waterfall repaired a distant galaxy through rigorous testing.",
    "The persistent researcher illuminated the dynamic equilibrium despite the inherent risks.",
    "The persistent researcher shattered the radioactive isotope for the sake of progress.",
    "A curious child analyzed the historical archive with remarkable efficiency.",
    "The seasoned astronaut manipulated a hidden frequency in absolute silence.",
    "The seasoned astronaut orchestrated the structural integrity for future generations.",
    "A quantum computer navigated a fading memory for future generations.",
    "A rogue asteroid bypassed an intricate puzzle using a novel approach.",
    "A curious child constructed the historical archive for future generations.",
    "The majestic eagle protected an intricate puzzle with reckless abandon.",
    "The majestic eagle calibrated a rogue signal with subtle elegance.",
    "A solitary lighthouse neutralized the harmonic resonance with cautious optimism.",
    "The massive whale protected a virtual environment at the speed of light.",
    "A clever hacker analyzed the fundamental theorem at the speed of light.",
    "A curious child discovered the quantum state at the speed of light.",
    "A curious child neutralized a virtual environment with cautious optimism.",
    "A hidden waterfall evaluated a philosophical concept using advanced techniques.",
    "The sprawling city bypassed the neural network through rigorous testing.",
    "The relentless storm investigated the harmonic resonance with astonishing precision.",
    "A solitary lighthouse embraced a mechanical flaw for the sake of progress.",
    "The persistent researcher investigated a virtual environment with mathematical certainty.",
    "A hidden waterfall embraced a digital footprint in perfect harmony.",
    "The ancient philosopher generated the historical archive using advanced techniques.",
    "A mysterious artifact investigated the complex algorithm against all odds.",
    "The majestic eagle highlighted a celestial body at the speed of light.",
    "The weary traveler neutralized the temporal anomaly with remarkable efficiency.",
    "A clever hacker amplified the neural network with remarkable efficiency.",
    "A brilliant detective observed the magnetic field during the final phase.",
    "The sprawling city highlighted a rogue signal in perfect harmony.",
    "The mechanical clock amplified a delicate ecosystem with astonishing precision.",
    "The massive whale manipulated the encrypted data despite the inherent risks.",
    "The relentless storm discovered the historical archive with remarkable efficiency.",
    "A rogue asteroid dismantled a philosophical concept using a novel approach.",
    "The forgotten king constructed a philosophical concept in absolute silence.",
    "The sprawling city bypassed a philosophical concept under the cover of darkness.",
    "The silent observer repaired the harmonic resonance without any prior warning.",
    "A solitary lighthouse investigated the historical archive with mathematical certainty.",
    "The massive whale manipulated a rogue signal under the cover of darkness.",
    "A fragile butterfly challenged the structural integrity in a spectacular display.",
    "An eager student observed a fading memory with remarkable efficiency.",
    "The silent observer illuminated a classified document for the sake of progress.",
    "A quantum computer amplified a psychological barrier despite the inherent risks.",
    "The silent observer observed a digital footprint with devastating consequences.",
    "The mechanical clock highlighted a philosophical concept for future generations.",
    "The sprawling city neutralized the quantum state at the speed of light.",
    "The ancient philosopher neutralized a classified document in a spectacular display.",
    "A quantum computer repaired the ancient ruins with reckless abandon.",
    "The swift fox generated a logical paradox using a novel approach.",
    "The silent observer repaired a delicate ecosystem with subtle elegance.",
    "A fragile butterfly evaluated the quantum state with astonishing precision.",
    "A fragile butterfly navigated the encrypted data in perfect harmony.",
    "The weary traveler embraced the harmonic resonance in absolute silence.",
    "A fragile butterfly analyzed the radioactive isotope with remarkable efficiency.",
    "The distant supernova embraced the neural network without any prior warning.",
    "The sprawling city evaluated a celestial body in a spectacular display.",
    "A clever hacker challenged the dynamic equilibrium using a novel approach.",
    "The gentle healer decoded the temporal anomaly beyond expected parameters.",
    "A brilliant detective dismantled a philosophical concept with mathematical certainty.",
    "A glowing embers discovered the magnetic field without hesitation.",
    "The distant supernova constructed the harmonic resonance with astonishing precision.",
    "A mysterious artifact orchestrated the structural integrity with devastating consequences.",
    "The seasoned astronaut bypassed a rogue signal using unprecedented methods.",
    "The relentless storm shattered the complex algorithm with devastating consequences.",
    "An autonomous robot navigated the magnetic field using a novel approach.",
    "The majestic eagle navigated the atmospheric pressure with remarkable efficiency.",
    "The swift fox protected the temporal anomaly without hesitation.",
    "The swift fox calibrated a logical paradox through rigorous testing.",
    "The weary traveler calculated a fading memory for the sake of progress.",
    "The sprawling city neutralized a philosophical concept without any prior warning.",
    "A wandering musician evaluated a digital footprint without any prior warning.",
    "A fierce gladiator synthesized a psychological barrier using advanced techniques.",
    "The silent observer amplified the complex algorithm using advanced techniques.",
    "The loyal companion shattered the ancient ruins in complete defiance.",
    "A quantum computer synthesized a distant galaxy through sheer willpower.",
    "The mechanical clock bypassed a celestial body without leaving a trace.",
    "An autonomous robot illuminated the temporal anomaly without any prior warning.",
    "A fierce gladiator synthesized the harmonic resonance using a novel approach.",
    "The sprawling city challenged a logical paradox without leaving a trace.",
    "A brilliant detective illuminated the temporal anomaly without hesitation.",
    "A curious child synthesized the ancient ruins in perfect harmony.",
    "An autonomous robot calibrated a distant galaxy in perfect harmony.",
    "A clever hacker embraced the atmospheric pressure with mathematical certainty.",
    "The gentle healer illuminated a logical paradox before the deadline.",
    "A fragile butterfly measured a microscopic organism with devastating consequences.",
    "A quantum computer orchestrated the fundamental theorem in complete defiance.",
    "The massive whale manipulated a distant galaxy for future generations.",
    "The weary traveler constructed the neural network in absolute silence.",
    "The persistent researcher generated a rogue signal during the final phase.",
    "The forgotten king protected the neural network using advanced techniques.",
    "A hidden waterfall measured the ancient ruins despite the inherent risks.",
    "A curious child investigated a celestial body without any prior warning.",
    "A fierce gladiator abandoned a mechanical flaw with devastating consequences.",
    "The silent observer highlighted the dynamic equilibrium through sheer willpower.",
    "The loyal companion bypassed the ancient ruins through rigorous testing.",
    "The loyal companion decoded a logical paradox at the speed of light.",
    "The seasoned astronaut neutralized a fading memory despite the inherent risks.",
    "A glowing embers overcame the ancient ruins for future generations.",
    "A clever hacker neutralized the radioactive isotope with subtle elegance.",
    "The distant supernova highlighted the encrypted data without hesitation.",
    "An elegant dancer protected the kinetic energy with astonishing precision.",
    "The relentless storm observed the structural integrity for the sake of progress.",
    "The massive whale analyzed the temporal anomaly using unprecedented methods.",
    "A fierce gladiator calculated the harmonic resonance during the final phase.",
    "The persistent researcher observed the radioactive isotope using unprecedented methods.",
    "A rogue asteroid orchestrated a logical paradox in absolute silence.",
    "A fragile butterfly measured a fading memory with subtle elegance.",
    "A quantum computer shattered the radioactive isotope in a controlled environment.",
    "The sprawling city illuminated the magnetic field using advanced techniques.",
    "A curious child dismantled a fading memory with reckless abandon.",
    "The forgotten king simulated the ancient ruins with remarkable efficiency.",
    "An eager student transformed the dynamic equilibrium using advanced techniques.",
    "The persistent researcher transformed a psychological barrier through sheer willpower.",
    "The persistent researcher orchestrated a psychological barrier with astonishing precision.",
    "A rogue asteroid illuminated a hidden frequency with mathematical certainty.",
    "A fierce gladiator calibrated the harmonic resonance using unprecedented methods.",
    "A mysterious artifact neutralized the atmospheric pressure for the sake of progress.",
    "A mysterious artifact navigated the encrypted data with reckless abandon.",
    "The ancient philosopher abandoned the structural integrity in complete defiance.",
    "A clever hacker analyzed the historical archive in absolute silence.",
    "The forgotten king simulated the neural network against all odds.",
    "The loyal companion dismantled a delicate ecosystem without any prior warning.",
    "The ancient philosopher repaired the kinetic energy using a novel approach.",
    "The silent observer highlighted the encrypted data with mathematical certainty.",
    "A fierce gladiator constructed an intricate puzzle in a controlled environment.",
    "The sprawling city synthesized an intricate puzzle in complete defiance.",
    "The majestic eagle amplified the historical archive for the sake of progress.",
    "An autonomous robot repaired a hidden frequency through sheer willpower.",
    "The distant supernova manipulated a distant galaxy in absolute silence.",
    "The mechanical clock analyzed the ancient ruins for future generations.",
    "A glowing embers illuminated a psychological barrier before the deadline.",
    "A brilliant detective discovered the complex algorithm in the blink of an eye.",
    "The distant supernova abandoned the structural integrity for the sake of progress.",
    "An autonomous robot simulated the neural network during the final phase.",
    "A rogue asteroid generated a microscopic organism in the blink of an eye.",
    "A wandering musician orchestrated an intricate puzzle in the blink of an eye.",
    "The forgotten king abandoned a logical paradox in perfect harmony.",
    "The sprawling city orchestrated a logical paradox in perfect harmony.",
    "A hidden waterfall analyzed the historical archive with mathematical certainty.",
    "A quantum computer synthesized the dynamic equilibrium without any prior warning.",
    "The silent observer decoded the temporal anomaly in the blink of an eye.",
    "The weary traveler constructed a delicate ecosystem before the deadline.",
    "A glowing embers transformed a celestial body before the deadline.",
    "An autonomous robot calculated the magnetic field without any prior warning.",
    "The persistent researcher analyzed the radioactive isotope in a controlled environment.",
    "The loyal companion repaired a philosophical concept with cautious optimism.",
    "A quantum computer abandoned the kinetic energy before the deadline.",
    "The sprawling city challenged the complex algorithm with subtle elegance.",
    "The forgotten king navigated a microscopic organism with cautious optimism.",
    "The ancient philosopher calculated a distant galaxy against all odds.",
    "The silent observer measured the harmonic resonance using unprecedented methods.",
    "The weary traveler embraced a rogue signal with devastating consequences.",
    "A clever hacker challenged the neural network with cautious optimism.",
    "The forgotten king shattered a digital footprint using unprecedented methods.",
    "The mechanical clock observed a microscopic organism with subtle elegance.",
    "A clever hacker abandoned the dynamic equilibrium under the cover of darkness.",
    "The gentle healer abandoned a classified document beyond expected parameters.",
    "A fierce gladiator observed a hidden frequency in a spectacular display.",
    "The majestic eagle navigated a celestial body beyond expected parameters.",
    "The silent observer overcame the dynamic equilibrium for future generations.",
    "A glowing embers discovered an intricate puzzle in perfect harmony.",
    "A rogue asteroid simulated the encrypted data under the cover of darkness.",
    "The gentle healer analyzed the temporal anomaly beyond expected parameters.",
    "A rogue asteroid transformed an intricate puzzle with subtle elegance.",
    "A clever hacker measured the fundamental theorem through rigorous testing.",
    "A mysterious artifact calibrated the dynamic equilibrium without any prior warning.",
    "The gentle healer repaired a celestial body in perfect harmony.",
    "A glowing embers evaluated a classified document without any prior warning.",
    "The sprawling city calculated a fading memory at the speed of light.",
    "A wandering musician manipulated the structural integrity without leaving a trace.",
    "The gentle healer simulated a hidden frequency for the sake of progress.",
    "The gentle healer challenged a virtual environment in perfect harmony.",
    "The forgotten king challenged a celestial body for the sake of progress.",
    "The mechanical clock decoded a logical paradox in a controlled environment.",
    "The mechanical clock embraced a hidden frequency with mathematical certainty.",
    "The sprawling city highlighted a virtual environment beyond expected parameters.",
    "The persistent researcher neutralized the neural network before the deadline.",
    "The majestic eagle overcame a distant galaxy in the blink of an eye.",
    "A mysterious artifact synthesized a delicate ecosystem through rigorous testing.",
    "The distant supernova evaluated a celestial body beyond expected parameters.",
    "A glowing embers calculated the ancient ruins in a controlled environment.",
    "A wandering musician abandoned a rogue signal through rigorous testing.",
    "An elegant dancer calibrated the fundamental theorem using advanced techniques.",
    "A quantum computer embraced the dynamic equilibrium without leaving a trace.",
    "The seasoned astronaut evaluated the neural network in perfect harmony.",
    "The loyal companion challenged a logical paradox using a novel approach.",
    "A hidden waterfall orchestrated a distant galaxy in a controlled environment.",
    "A wandering musician generated the encrypted data in a controlled environment.",
    "The ancient philosopher protected the historical archive beyond expected parameters.",
    "The ancient philosopher amplified a digital footprint with subtle elegance.",
    "The silent observer dismantled a philosophical concept at the speed of light.",
    "The silent observer amplified the neural network for future generations.",
    "A brilliant detective challenged an intricate puzzle at the speed of light.",
    "The massive whale calculated the neural network before the deadline.",
    "A wandering musician synthesized the dynamic equilibrium at the speed of light.",
    "The swift fox calculated a logical paradox with cautious optimism.",
    "The ancient philosopher illuminated a classified document despite the inherent risks.",
    "An autonomous robot highlighted a digital footprint against all odds.",
    "An elegant dancer synthesized the neural network without hesitation.",
    "A solitary lighthouse orchestrated a logical paradox in complete defiance.",
    "An autonomous robot protected the quantum state with remarkable efficiency.",
    "The weary traveler evaluated a digital footprint with devastating consequences.",
    "A quantum computer calculated a classified document under the cover of darkness.",
    "A clever hacker constructed the complex algorithm during the final phase.",
    "The seasoned astronaut manipulated a hidden frequency beyond expected parameters.",
    "The sprawling city challenged a microscopic organism using advanced techniques.",
    "A solitary lighthouse overcame the historical archive through rigorous testing.",
    "A curious child generated a mechanical flaw during the final phase.",
    "The sprawling city observed a hidden frequency in a spectacular display.",
    "The seasoned astronaut orchestrated a fading memory using advanced techniques.",
    "The silent observer neutralized a philosophical concept for future generations.",
    "A quantum computer abandoned the magnetic field with subtle elegance.",
    "A quantum computer protected the encrypted data under the cover of darkness.",
    "A rogue asteroid amplified a philosophical concept without any prior warning.",
    "A rogue asteroid calculated the magnetic field using a novel approach.",
    "A rogue asteroid challenged a rogue signal with devastating consequences.",
    "The sprawling city evaluated the historical archive in absolute silence.",
    "The silent observer observed an intricate puzzle with reckless abandon.",
    "A wandering musician shattered the complex algorithm in perfect harmony.",
    "The majestic eagle neutralized the neural network using a novel approach.",
    "A fierce gladiator investigated an intricate puzzle beyond expected parameters.",
    "A rogue asteroid abandoned a logical paradox at the speed of light.",
    "The sprawling city analyzed a digital footprint with reckless abandon.",
    "The loyal companion synthesized the encrypted data with devastating consequences.",
    "A fierce gladiator overcame the complex algorithm with subtle elegance.",
    "The ancient philosopher calibrated the harmonic resonance in the blink of an eye.",
    "A fierce gladiator shattered the harmonic resonance with reckless abandon.",
    "The distant supernova synthesized a rogue signal with astonishing precision.",
    "The seasoned astronaut repaired a virtual environment despite the inherent risks.",
    "The forgotten king protected the magnetic field with remarkable efficiency.",
    "The persistent researcher challenged a philosophical concept without any prior warning.",
    "A glowing embers bypassed a fading memory using a novel approach.",
    "A quantum computer investigated the atmospheric pressure under the cover of darkness.",
    "An elegant dancer illuminated the structural integrity under the cover of darkness.",
    "The persistent researcher analyzed a psychological barrier at the speed of light.",
    "A fierce gladiator generated the dynamic equilibrium in a spectacular display.",
    "A solitary lighthouse overcame the dynamic equilibrium with devastating consequences.",
    "The sprawling city measured the encrypted data without any prior warning.",
    "The sprawling city simulated the temporal anomaly using advanced techniques.",
    "The loyal companion manipulated a mechanical flaw with cautious optimism.",
    "The ancient philosopher abandoned the encrypted data without hesitation.",
    "The ancient philosopher simulated the harmonic resonance without any prior warning.",
    "A mysterious artifact analyzed the ancient ruins without hesitation.",
    "The silent observer analyzed a hidden frequency in absolute silence.",
    "A fierce gladiator decoded the complex algorithm using a novel approach.",
    "A rogue asteroid generated a distant galaxy under the cover of darkness.",
    "A clever hacker synthesized the neural network for the sake of progress.",
    "The sprawling city transformed a microscopic organism despite the inherent risks.",
    "The weary traveler observed the magnetic field with astonishing precision.",
    "The gentle healer challenged the fundamental theorem in the blink of an eye.",
    "The massive whale measured a digital footprint in absolute silence.",
    "The swift fox measured the atmospheric pressure with subtle elegance.",
    "The distant supernova manipulated the harmonic resonance with mathematical certainty.",
    "The silent observer simulated a hidden frequency through rigorous testing.",
    "An elegant dancer bypassed the structural integrity in a spectacular display.",
    "A glowing embers embraced the magnetic field for the sake of progress.",
    "The massive whale investigated the quantum state with subtle elegance.",
    "The distant supernova protected the complex algorithm with devastating consequences.",
    "The forgotten king neutralized a celestial body during the final phase.",
    "The massive whale simulated a fading memory in complete defiance.",
    "The loyal companion generated a mechanical flaw without leaving a trace.",
    "The sprawling city orchestrated a fading memory before the deadline.",
    "The weary traveler illuminated the harmonic resonance in absolute silence.",
    "The distant supernova calculated a philosophical concept with astonishing precision.",
    "An elegant dancer calculated a celestial body with remarkable efficiency.",
    "A rogue asteroid bypassed a digital footprint with devastating consequences.",
    "The sprawling city protected the kinetic energy in perfect harmony.",
    "A mysterious artifact discovered a delicate ecosystem for the sake of progress.",
    "An elegant dancer simulated the harmonic resonance using unprecedented methods.",
    "The ancient philosopher investigated the atmospheric pressure beyond expected parameters.",
    "The majestic eagle evaluated the temporal anomaly under the cover of darkness.",
    "A wandering musician measured a microscopic organism with subtle elegance.",
    "A fragile butterfly investigated the historical archive in the blink of an eye.",
    "The weary traveler analyzed a logical paradox despite the inherent risks.",
    "The swift fox decoded the ancient ruins with devastating consequences.",
    "The silent observer repaired a distant galaxy at the speed of light.",
    "The swift fox illuminated a mechanical flaw for future generations.",
    "A wandering musician calculated the harmonic resonance for future generations.",
    "The relentless storm manipulated the fundamental theorem without leaving a trace.",
    "The massive whale navigated the ancient ruins during the final phase.",
    "A mysterious artifact orchestrated the fundamental theorem with subtle elegance.",
    "A fierce gladiator investigated a classified document during the final phase.",
    "The weary traveler calculated the ancient ruins before the deadline.",
    "The seasoned astronaut abandoned the quantum state without hesitation.",
    "The ancient philosopher orchestrated a hidden frequency in perfect harmony.",
    "The distant supernova generated the temporal anomaly with remarkable efficiency.",
    "A hidden waterfall transformed a fading memory without any prior warning.",
    "The relentless storm calculated the dynamic equilibrium against all odds.",
    "A rogue asteroid calculated a microscopic organism against all odds.",
    "The swift fox illuminated a classified document with reckless abandon.",
    "A rogue asteroid neutralized a psychological barrier with astonishing precision.",
    "A rogue asteroid investigated the magnetic field through rigorous testing.",
    "The relentless storm investigated a classified document with devastating consequences.",
    "The sprawling city dismantled a philosophical concept with astonishing precision.",
    "A hidden waterfall repaired a mechanical flaw without any prior warning.",
    "A wandering musician calibrated the ancient ruins through sheer willpower.",
    "The ancient philosopher evaluated a hidden frequency in a controlled environment.",
    "A brilliant detective embraced a digital footprint using advanced techniques.",
    "The mechanical clock observed a celestial body using advanced techniques.",
    "The relentless storm simulated a distant galaxy during the final phase.",
    "The distant supernova navigated the quantum state without hesitation.",
    "A clever hacker investigated the magnetic field using a novel approach.",
    "A rogue asteroid investigated the historical archive in absolute silence.",
    "The seasoned astronaut synthesized the quantum state in the blink of an eye.",
    "A mysterious artifact challenged a mechanical flaw in a controlled environment.",
    "The seasoned astronaut dismantled a hidden frequency with mathematical certainty.",
    "A fragile butterfly calculated a classified document despite the inherent risks.",
    "A quantum computer shattered the quantum state despite the inherent risks.",
    "The persistent researcher decoded the quantum state through sheer willpower.",
    "The forgotten king illuminated the ancient ruins in the blink of an eye.",
    "The relentless storm calibrated a hidden frequency through rigorous testing.",
    "A rogue asteroid overcame a microscopic organism using a novel approach.",
    "A fragile butterfly illuminated the temporal anomaly in perfect harmony.",
    "The loyal companion navigated a philosophical concept with cautious optimism.",
    "The relentless storm amplified the kinetic energy for future generations.",
    "The relentless storm investigated the radioactive isotope in absolute silence.",
    "A curious child abandoned the dynamic equilibrium with astonishing precision.",
    "A clever hacker navigated a philosophical concept beyond expected parameters.",
    "A wandering musician highlighted the ancient ruins without hesitation.",
    "The sprawling city discovered a hidden frequency with reckless abandon.",
    "A wandering musician neutralized the complex algorithm for future generations.",
    "A quantum computer discovered a psychological barrier in the blink of an eye.",
    "A curious child calibrated a classified document through rigorous testing.",
    "The ancient philosopher illuminated the encrypted data before the deadline.",
    "A brilliant detective abandoned the quantum state without hesitation.",
    "A brilliant detective synthesized the neural network in a controlled environment.",
    "A hidden waterfall observed a distant galaxy against all odds.",
    "The forgotten king decoded the encrypted data beyond expected parameters.",
    "A curious child analyzed the complex algorithm with mathematical certainty.",
    "The sprawling city generated a logical paradox with astonishing precision.",
    "A curious child synthesized the atmospheric pressure despite the inherent risks.",
    "A glowing embers neutralized a virtual environment during the final phase.",
    "The weary traveler highlighted a distant galaxy in the blink of an eye.",
    "The persistent researcher repaired the neural network with mathematical certainty.",
    "The relentless storm evaluated the dynamic equilibrium with subtle elegance.",
    "A brilliant detective observed an intricate puzzle through rigorous testing.",
    "A solitary lighthouse abandoned the radioactive isotope in complete defiance.",
    "The seasoned astronaut illuminated a mechanical flaw without any prior warning.",
    "The relentless storm calibrated the kinetic energy before the deadline.",
    "The massive whale calibrated a mechanical flaw for the sake of progress.",
    "The silent observer calculated the neural network without leaving a trace.",
    "A wandering musician orchestrated the fundamental theorem with reckless abandon.",
    "A fierce gladiator decoded the quantum state with reckless abandon.",
    "An elegant dancer dismantled a mechanical flaw during the final phase.",
    "A glowing embers synthesized a mechanical flaw through sheer willpower.",
    "The weary traveler highlighted the radioactive isotope using unprecedented methods.",
    "The ancient philosopher observed the encrypted data using a novel approach.",
    "A mysterious artifact challenged a classified document without any prior warning.",
    "The silent observer overcame the kinetic energy in complete defiance.",
    "The gentle healer measured the encrypted data with mathematical certainty.",
    "A rogue asteroid constructed the complex algorithm in complete defiance.",
    "A fierce gladiator calibrated a psychological barrier without hesitation.",
    "The forgotten king protected a celestial body for the sake of progress.",
    "A clever hacker investigated a rogue signal in a spectacular display.",
    "A curious child amplified the historical archive with astonishing precision.",
    "The massive whale generated the quantum state at the speed of light.",
    "A fierce gladiator calibrated a psychological barrier against all odds.",
    "A curious child shattered the radioactive isotope despite the inherent risks.",
    "A solitary lighthouse shattered the kinetic energy in the blink of an eye.",
    "A hidden waterfall highlighted the fundamental theorem for future generations.",
    "An eager student calibrated the dynamic equilibrium with remarkable efficiency.",
    "A clever hacker investigated a fading memory with astonishing precision.",
    "The ancient philosopher simulated the encrypted data without leaving a trace.",
    "A clever hacker calculated a microscopic organism in complete defiance.",
    "A solitary lighthouse highlighted a fading memory using unprecedented methods.",
    "A fierce gladiator observed the kinetic energy despite the inherent risks.",
    "A fragile butterfly amplified a classified document for the sake of progress.",
    "The distant supernova repaired a rogue signal in a spectacular display.",
    "A mysterious artifact orchestrated a distant galaxy with mathematical certainty.",
    "An autonomous robot observed the temporal anomaly against all odds.",
    "A quantum computer observed the fundamental theorem during the final phase.",
    "The persistent researcher protected a philosophical concept with astonishing precision.",
    "The ancient philosopher neutralized a digital footprint through sheer willpower.",
    "A fragile butterfly transformed the harmonic resonance through sheer willpower.",
    "The swift fox calibrated a logical paradox without hesitation.",
    "A hidden waterfall analyzed a psychological barrier without any prior warning.",
    "A curious child discovered a digital footprint without hesitation.",
    "The majestic eagle decoded the atmospheric pressure through rigorous testing.",
    "The distant supernova bypassed a classified document at the speed of light.",
    "The swift fox protected the magnetic field with mathematical certainty.",
    "The sprawling city simulated a fading memory during the final phase.",
    "A fragile butterfly synthesized the quantum state in a spectacular display.",
    "The majestic eagle transformed a classified document in absolute silence.",
    "The sprawling city measured the radioactive isotope with mathematical certainty.",
    "A rogue asteroid simulated the radioactive isotope in a spectacular display.",
    "A brilliant detective calculated the dynamic equilibrium in a controlled environment.",
    "The mechanical clock evaluated a philosophical concept in perfect harmony.",
    "A brilliant detective calibrated a distant galaxy through rigorous testing.",
    "A hidden waterfall overcame an intricate puzzle for future generations.",
    "A brilliant detective neutralized the structural integrity beyond expected parameters.",
    "The mechanical clock observed a microscopic organism in complete defiance.",
    "The loyal companion navigated a virtual environment with reckless abandon.",
    "A solitary lighthouse bypassed the encrypted data despite the inherent risks.",
    "A rogue asteroid highlighted a classified document in a controlled environment.",
    "A hidden waterfall transformed the fundamental theorem beyond expected parameters.",
    "A fragile butterfly generated the historical archive through sheer willpower.",
    "The ancient philosopher simulated the encrypted data under the cover of darkness.",
    "The sprawling city manipulated a philosophical concept with remarkable efficiency.",
    "The majestic eagle analyzed the dynamic equilibrium in absolute silence.",
    "The relentless storm neutralized an intricate puzzle using a novel approach.",
    "The forgotten king amplified a psychological barrier with reckless abandon.",
    "A clever hacker measured the harmonic resonance without leaving a trace.",
    "The sprawling city analyzed a distant galaxy with reckless abandon.",
    "The seasoned astronaut synthesized the dynamic equilibrium in perfect harmony.",
    "The majestic eagle navigated a delicate ecosystem without any prior warning.",
    "The relentless storm calculated a hidden frequency with cautious optimism.",
    "A fierce gladiator synthesized the neural network without hesitation.",
    "A fragile butterfly neutralized the atmospheric pressure before the deadline.",
    "A glowing embers constructed a fading memory in the blink of an eye.",
    "The relentless storm observed the fundamental theorem under the cover of darkness.",
    "A quantum computer abandoned a psychological barrier in a spectacular display.",
    "A quantum computer challenged the quantum state despite the inherent risks.",
    "The swift fox synthesized a microscopic organism through rigorous testing.",
    "A curious child investigated a logical paradox in the blink of an eye.",
    "The weary traveler neutralized a fading memory through rigorous testing.",
    "A glowing embers highlighted a rogue signal in perfect harmony.",
    "The swift fox repaired the kinetic energy without hesitation.",
    "The persistent researcher protected the dynamic equilibrium without leaving a trace.",
    "An autonomous robot analyzed the structural integrity using advanced techniques.",
    "A curious child dismantled the magnetic field with cautious optimism.",
    "The gentle healer overcame the magnetic field with mathematical certainty.",
    "An elegant dancer measured the atmospheric pressure in absolute silence.",
    "A brilliant detective calculated a classified document at the speed of light.",
    "A solitary lighthouse repaired the kinetic energy with reckless abandon.",
    "The forgotten king constructed the harmonic resonance without hesitation.",
    "A fierce gladiator embraced a classified document under the cover of darkness.",
    "The loyal companion transformed a distant galaxy in perfect harmony.",
    "A glowing embers embraced a rogue signal through rigorous testing.",
    "A rogue asteroid protected a hidden frequency using unprecedented methods.",
    "A fierce gladiator investigated a psychological barrier despite the inherent risks.",
    "The silent observer analyzed an intricate puzzle in a controlled environment.",
    "The gentle healer repaired a hidden frequency through sheer willpower.",
    "The distant supernova decoded the quantum state with mathematical certainty.",
    "A fierce gladiator overcame the magnetic field for future generations.",
    "The massive whale calibrated the fundamental theorem with devastating consequences.",
    "A brilliant detective discovered a celestial body before the deadline.",
    "An elegant dancer constructed the complex algorithm in absolute silence.",
    "A brilliant detective measured a delicate ecosystem during the final phase.",
    "The relentless storm observed the ancient ruins without hesitation.",
    "A clever hacker challenged a delicate ecosystem in absolute silence.",
    "The loyal companion illuminated the neural network using a novel approach.",
    "The weary traveler observed the structural integrity with devastating consequences.",
    "A quantum computer abandoned a celestial body with mathematical certainty.",
    "A wandering musician calibrated the atmospheric pressure through sheer willpower.",
    "The sprawling city evaluated a logical paradox using unprecedented methods.",
]

EVAL_SENTENCES = [
    "The forgotten king highlighted a mechanical flaw in perfect harmony.",
    "The mechanical clock manipulated a delicate ecosystem using advanced techniques.",
    "A brilliant detective analyzed a logical paradox in perfect harmony.",
    "The massive whale calibrated the kinetic energy without any prior warning.",
    "The swift fox shattered the encrypted data with reckless abandon.",
    "An autonomous robot orchestrated the encrypted data in a spectacular display.",
    "The seasoned astronaut transformed the structural integrity for future generations.",
    "A hidden waterfall constructed the neural network beyond expected parameters.",
    "The sprawling city observed a distant galaxy under the cover of darkness.",
    "A brilliant detective manipulated the harmonic resonance in the blink of an eye.",
    "A clever hacker dismantled the ancient ruins in perfect harmony.",
    "A mysterious artifact embraced a philosophical concept against all odds.",
    "A brilliant detective overcame a hidden frequency with remarkable efficiency.",
    "A fierce gladiator analyzed the harmonic resonance through sheer willpower.",
    "The sprawling city neutralized an intricate puzzle in perfect harmony.",
    "A fierce gladiator measured a delicate ecosystem without hesitation.",
    "The gentle healer evaluated a distant galaxy beyond expected parameters.",
    "The loyal companion manipulated the temporal anomaly in the blink of an eye.",
    "A rogue asteroid dismantled a virtual environment in complete defiance.",
    "The weary traveler observed a hidden frequency for future generations.",
    "A solitary lighthouse calculated a philosophical concept in a controlled environment.",
    "A fierce gladiator embraced the encrypted data with mathematical certainty.",
    "A hidden waterfall manipulated a mechanical flaw through sheer willpower.",
    "A wandering musician measured the dynamic equilibrium under the cover of darkness.",
    "An elegant dancer calibrated the harmonic resonance with mathematical certainty.",
    "An elegant dancer challenged a virtual environment without any prior warning.",
    "The seasoned astronaut embraced a philosophical concept without hesitation.",
    "An elegant dancer overcame the quantum state against all odds.",
    "A wandering musician simulated the harmonic resonance through sheer willpower.",
    "A hidden waterfall simulated a hidden frequency with devastating consequences.",
    "A hidden waterfall neutralized a psychological barrier without hesitation.",
    "A glowing embers shattered a mechanical flaw in absolute silence.",
    "The swift fox illuminated the dynamic equilibrium through sheer willpower.",
    "The sprawling city protected the fundamental theorem with subtle elegance.",
    "A clever hacker illuminated the dynamic equilibrium without leaving a trace.",
    "The massive whale transformed the kinetic energy with devastating consequences.",
    "A brilliant detective generated a fading memory against all odds.",
    "The sprawling city discovered the radioactive isotope against all odds.",
    "A brilliant detective observed the dynamic equilibrium through sheer willpower.",
    "A rogue asteroid overcame a distant galaxy using unprecedented methods.",
    "The weary traveler analyzed the quantum state using unprecedented methods.",
    "The distant supernova evaluated the atmospheric pressure with reckless abandon.",
    "A clever hacker challenged a digital footprint without any prior warning.",
    "The loyal companion challenged the dynamic equilibrium through sheer willpower.",
    "A wandering musician calibrated the magnetic field with cautious optimism.",
    "The majestic eagle calculated a distant galaxy with mathematical certainty.",
    "A fragile butterfly amplified a philosophical concept in a controlled environment.",
    "A clever hacker dismantled a hidden frequency using a novel approach.",
    "The gentle healer shattered the temporal anomaly for future generations.",
    "A fierce gladiator protected the dynamic equilibrium in a controlled environment.",
    "The mechanical clock decoded the complex algorithm with remarkable efficiency.",
    "A fierce gladiator investigated the structural integrity in perfect harmony.",
    "The swift fox calculated the ancient ruins under the cover of darkness.",
    "A fragile butterfly decoded the kinetic energy before the deadline.",
    "The mechanical clock manipulated an intricate puzzle without leaving a trace.",
    "A rogue asteroid neutralized a psychological barrier at the speed of light.",
    "The relentless storm highlighted the quantum state using unprecedented methods.",
    "The ancient philosopher simulated an intricate puzzle under the cover of darkness.",
    "The distant supernova constructed the atmospheric pressure against all odds.",
    "An eager student investigated an intricate puzzle using unprecedented methods.",
    "The gentle healer dismantled the structural integrity in complete defiance.",
    "A glowing embers simulated the structural integrity through rigorous testing.",
    "The sprawling city navigated a microscopic organism beyond expected parameters.",
    "A quantum computer abandoned a classified document through sheer willpower.",
    "A hidden waterfall dismantled the complex algorithm in a controlled environment.",
    "The mechanical clock manipulated the quantum state through sheer willpower.",
    "The majestic eagle manipulated the radioactive isotope despite the inherent risks.",
    "The gentle healer bypassed the dynamic equilibrium despite the inherent risks.",
    "A glowing embers overcame the ancient ruins without any prior warning.",
    "The massive whale investigated the magnetic field through rigorous testing.",
    "The massive whale observed the historical archive with remarkable efficiency.",
    "A solitary lighthouse challenged the atmospheric pressure without leaving a trace.",
    "The sprawling city calculated the historical archive for future generations.",
    "The weary traveler simulated the complex algorithm with reckless abandon.",
    "The distant supernova evaluated a digital footprint in the blink of an eye.",
    "The distant supernova illuminated a psychological barrier in absolute silence.",
    "The seasoned astronaut measured a microscopic organism using unprecedented methods.",
    "A wandering musician highlighted a microscopic organism for future generations.",
    "The majestic eagle investigated the harmonic resonance with reckless abandon.",
    "The massive whale decoded a microscopic organism using unprecedented methods.",
    "A glowing embers synthesized the encrypted data despite the inherent risks.",
    "An elegant dancer neutralized the magnetic field using a novel approach.",
    "A clever hacker overcame the fundamental theorem through sheer willpower.",
    "A clever hacker generated a virtual environment with remarkable efficiency.",
    "The loyal companion generated the structural integrity with devastating consequences.",
    "The silent observer challenged the magnetic field without any prior warning.",
    "A fragile butterfly orchestrated the neural network with remarkable efficiency.",
    "The silent observer manipulated a philosophical concept using a novel approach.",
    "A brilliant detective constructed the neural network during the final phase.",
    "The massive whale evaluated the ancient ruins without leaving a trace.",
    "A glowing embers manipulated a microscopic organism in perfect harmony.",
    "A clever hacker bypassed the historical archive in complete defiance.",
    "The seasoned astronaut analyzed a rogue signal with subtle elegance.",
    "A quantum computer amplified a digital footprint with reckless abandon.",
    "A curious child repaired the temporal anomaly with devastating consequences.",
    "A fragile butterfly analyzed the ancient ruins in the blink of an eye.",
    "A fierce gladiator bypassed the encrypted data against all odds.",
    "The sprawling city synthesized a hidden frequency in absolute silence.",
    "The massive whale transformed a logical paradox using a novel approach.",
    "The relentless storm transformed a logical paradox using a novel approach.",
    "A fragile butterfly manipulated the magnetic field in absolute silence.",
    "The majestic eagle generated a hidden frequency using advanced techniques.",
    "A curious child overcame a classified document in absolute silence.",
    "The massive whale navigated the quantum state in perfect harmony.",
    "The mechanical clock calibrated a distant galaxy in absolute silence.",
    "An elegant dancer calibrated a digital footprint in complete defiance.",
    "A hidden waterfall calculated the dynamic equilibrium in absolute silence.",
    "A curious child protected the structural integrity using a novel approach.",
    "A curious child neutralized a fading memory at the speed of light.",
    "The forgotten king calculated the neural network with reckless abandon.",
    "The weary traveler analyzed a digital footprint with astonishing precision.",
    "The sprawling city embraced a delicate ecosystem at the speed of light.",
    "The silent observer evaluated the kinetic energy without leaving a trace.",
    "The loyal companion shattered the neural network against all odds.",
    "A fragile butterfly constructed a rogue signal in absolute silence.",
    "A solitary lighthouse observed the complex algorithm under the cover of darkness.",
    "The seasoned astronaut dismantled a mechanical flaw at the speed of light.",
    "The weary traveler shattered a microscopic organism through sheer willpower.",
    "A quantum computer embraced the radioactive isotope in complete defiance.",
    "The gentle healer transformed a rogue signal with cautious optimism.",
    "The persistent researcher shattered the atmospheric pressure without hesitation.",
    "A brilliant detective protected the structural integrity beyond expected parameters.",
    "The persistent researcher shattered the complex algorithm beyond expected parameters.",
    "A curious child measured the fundamental theorem using advanced techniques.",
    "The sprawling city embraced the quantum state through sheer willpower.",
    "The massive whale investigated a digital footprint with mathematical certainty.",
    "A rogue asteroid protected the atmospheric pressure with reckless abandon.",
    "A fierce gladiator transformed the structural integrity in the blink of an eye.",
    "A wandering musician observed the kinetic energy with reckless abandon.",
    "The swift fox amplified the complex algorithm beyond expected parameters.",
    "The massive whale neutralized a celestial body without any prior warning.",
    "A mysterious artifact orchestrated the radioactive isotope under the cover of darkness.",
    "The forgotten king embraced an intricate puzzle against all odds.",
    "The massive whale orchestrated the magnetic field at the speed of light.",
    "A rogue asteroid investigated the encrypted data against all odds.",
    "A fierce gladiator observed a fading memory in complete defiance.",
    "An elegant dancer calibrated a psychological barrier beyond expected parameters.",
    "The ancient philosopher constructed the ancient ruins for future generations.",
    "A solitary lighthouse measured the neural network during the final phase.",
    "The massive whale challenged the ancient ruins without any prior warning.",
    "The massive whale dismantled a microscopic organism with remarkable efficiency.",
    "A fierce gladiator repaired the atmospheric pressure using advanced techniques.",
    "The relentless storm simulated a delicate ecosystem during the final phase.",
    "The silent observer decoded a virtual environment in the blink of an eye.",
    "An autonomous robot manipulated a logical paradox through rigorous testing.",
    "The seasoned astronaut neutralized a fading memory against all odds.",
    "The distant supernova challenged a logical paradox in the blink of an eye.",
    "A solitary lighthouse constructed a distant galaxy under the cover of darkness.",
    "The loyal companion investigated a logical paradox beyond expected parameters.",
    "A hidden waterfall investigated a rogue signal without any prior warning.",
    "A quantum computer analyzed a celestial body before the deadline.",
    "The seasoned astronaut calibrated the complex algorithm with remarkable efficiency.",
    "An autonomous robot discovered the historical archive through rigorous testing.",
    "A clever hacker embraced the quantum state for future generations.",
    "The distant supernova abandoned a delicate ecosystem without leaving a trace.",
    "The mechanical clock amplified the ancient ruins against all odds.",
    "An autonomous robot dismantled a mechanical flaw with subtle elegance.",
    "The gentle healer bypassed the complex algorithm before the deadline.",
    "A brilliant detective shattered the kinetic energy with reckless abandon.",
    "The sprawling city calculated the dynamic equilibrium in complete defiance.",
    "A quantum computer amplified the temporal anomaly in a spectacular display.",
    "The mechanical clock calculated the dynamic equilibrium without leaving a trace.",
    "A wandering musician investigated an intricate puzzle with devastating consequences.",
    "A fierce gladiator bypassed the historical archive in a controlled environment.",
    "The majestic eagle highlighted the temporal anomaly without hesitation.",
    "The majestic eagle observed the dynamic equilibrium under the cover of darkness.",
    "The massive whale bypassed a fading memory using unprecedented methods.",
    "A glowing embers highlighted the structural integrity at the speed of light.",
    "A hidden waterfall calculated the fundamental theorem before the deadline.",
    "A clever hacker simulated the fundamental theorem with devastating consequences.",
    "A curious child investigated the neural network in perfect harmony.",
    "A clever hacker highlighted a rogue signal against all odds.",
    "A glowing embers observed the kinetic energy with subtle elegance.",
    "The mechanical clock neutralized a philosophical concept with cautious optimism.",
    "A quantum computer bypassed a classified document with cautious optimism.",
    "A solitary lighthouse neutralized an intricate puzzle under the cover of darkness.",
    "A fragile butterfly synthesized the dynamic equilibrium during the final phase.",
    "An elegant dancer navigated a psychological barrier using advanced techniques.",
    "An elegant dancer amplified a microscopic organism before the deadline.",
    "The weary traveler protected the complex algorithm with mathematical certainty.",
    "The persistent researcher abandoned a philosophical concept using advanced techniques.",
    "The sprawling city evaluated the radioactive isotope without any prior warning.",
    "The persistent researcher embraced a classified document in complete defiance.",
    "A fragile butterfly dismantled a digital footprint without hesitation.",
    "A rogue asteroid illuminated the quantum state at the speed of light.",
    "A wandering musician evaluated the encrypted data through rigorous testing.",
    "The gentle healer decoded a mechanical flaw with remarkable efficiency.",
    "The seasoned astronaut calibrated the complex algorithm through rigorous testing.",
    "The mechanical clock generated a classified document through sheer willpower.",
    "An eager student manipulated the temporal anomaly through rigorous testing.",
    "A quantum computer embraced a rogue signal using advanced techniques.",
    "The majestic eagle transformed a delicate ecosystem during the final phase.",
    "An eager student repaired a rogue signal before the deadline.",
    "A wandering musician dismantled the quantum state beyond expected parameters.",
    "The swift fox bypassed the temporal anomaly against all odds.",
    "The sprawling city navigated the radioactive isotope with reckless abandon.",
    "The silent observer generated the neural network against all odds.",
    "The swift fox decoded the harmonic resonance before the deadline.",
    "The seasoned astronaut generated the ancient ruins for future generations.",
    "The distant supernova orchestrated a logical paradox in a controlled environment.",
]

if MAX_TRAIN: TRAIN_SENTENCES = TRAIN_SENTENCES[:MAX_TRAIN]
if MAX_EVAL:  EVAL_SENTENCES = EVAL_SENTENCES[:MAX_EVAL]
print("train:", len(TRAIN_SENTENCES), "| eval:", len(EVAL_SENTENCES))

## Grammar parse, relations, and word->token mapping

Roles are read off by word index: `DET(0) ADJ(1) NOUN(2) VERB(3) DET(4) ADJ(5) NOUN(6)`. `build_word_spans` reconstructs each whitespace word's token indices from the BPE pieces (a new word begins at a leading-space token), accounting for the BOS token at index 0. Relations are `attender_widx -> receiver_widx`.

In [ ]:
# attender word index -> receiver word index (the receiver is what the attender should attend to)
RELATIONS = [
    {"name": "object_to_verb",      "attender_widx": 6, "receiver_widx": 3},  # obj noun  -> verb
    {"name": "subject_to_verb",     "attender_widx": 2, "receiver_widx": 3},  # subj noun -> verb
    {"name": "object_adj_to_noun",  "attender_widx": 5, "receiver_widx": 6},  # obj adj   -> obj noun
    {"name": "subject_adj_to_noun", "attender_widx": 1, "receiver_widx": 2},  # subj adj  -> subj noun
    {"name": "object_det_to_noun",  "attender_widx": 4, "receiver_widx": 6},  # obj det   -> obj noun
    {"name": "subject_det_to_noun", "attender_widx": 0, "receiver_widx": 2},  # subj det  -> subj noun
]
MAX_WIDX = max(max(r["attender_widx"], r["receiver_widx"]) for r in RELATIONS)

DETERMINERS = {"a", "an", "the"}

def valid_template(sent):
    w = sent.strip().rstrip(".").split()
    return len(w) >= 7 and w[0].lower() in DETERMINERS and w[4].lower() in DETERMINERS

def build_word_spans(readable_tokens, include_bos=True):
    """Reconstruct, for each whitespace word, the list of token indices covering it."""
    spans = []; cur = []
    start = 1 if include_bos else 0      # skip BOS at index 0
    for i in range(start, len(readable_tokens)):
        piece = readable_tokens[i]
        if (piece[:1] in (" ", "\t", "\n")) or not cur:
            if cur: spans.append(cur)
            cur = [i]
        else:
            cur.append(i)
    if cur: spans.append(cur)
    return spans

# sanity check on one sentence
_att, _toks, _vis, _ust = get_all_attention_matrices_at_time(
    model=model, tokenizer=tokenizer, text_sequence=TRAIN_SENTENCES[0],
    diffusion_time=FINAL_TIME, diffusion_steps=DIFFUSION_STEPS, seed=SEED, include_bos=True)
_spans = build_word_spans(_toks, include_bos=True)
print("sentence:", TRAIN_SENTENCES[0])
for wi in range(MAX_WIDX + 1):
    print(f"  word[{wi}] ->", [_toks[t] for t in _spans[wi]])

## Score every head on a dataset

For each valid sentence and relation, the attender row's masked `argmax` must land on a receiver token. Accuracy is accumulated per `(layer, head, relation)`.

In [ ]:
def score_dataset(sentences, tag=""):
    n_layers = n_heads = None
    corr = {r["name"]: None for r in RELATIONS}
    tot = {r["name"]: 0 for r in RELATIONS}
    used = skipped = 0
    for si, sent in enumerate(sentences):
        if not valid_template(sent):
            skipped += 1; continue
        attentions, toks, vis, ust = get_all_attention_matrices_at_time(
            model=model, tokenizer=tokenizer, text_sequence=sent,
            diffusion_time=FINAL_TIME, diffusion_steps=DIFFUSION_STEPS, seed=SEED, include_bos=True)
        if n_layers is None:
            n_layers = len(attentions); n_heads = attentions[0].shape[1]
            for r in RELATIONS:
                corr[r["name"]] = np.zeros((n_layers, n_heads), dtype=np.float64)
        spans = build_word_spans(toks, include_bos=True)
        if len(spans) <= MAX_WIDX:
            skipped += 1; continue
        used += 1
        for r in RELATIONS:
            aspan = spans[r["attender_widx"]]
            rset = set(spans[r["receiver_widx"]])
            ar = aspan[-1] if ATTENDER_TOKEN == "last" else aspan[0]
            tot[r["name"]] += 1
            for l in range(n_layers):
                row = attentions[l][0, :, ar, :].detach().float().clone()   # [heads, S]
                if EXCLUDE_BOS:
                    row[:, 0] = float("-inf")
                if EXCLUDE_SELF:
                    for c in aspan:
                        row[:, c] = float("-inf")
                pred = row.argmax(dim=1).cpu().numpy()                       # [heads]
                corr[r["name"]][l] += np.fromiter((p in rset for p in pred), dtype=np.float64, count=n_heads)
        if (si + 1) % 200 == 0:
            print(f"  [{tag}] {si + 1}/{len(sentences)} processed")
    rows = []
    for r in RELATIONS:
        if corr[r["name"]] is None:
            continue
        acc = corr[r["name"]] / max(tot[r["name"]], 1)
        for l in range(acc.shape[0]):
            for h in range(acc.shape[1]):
                rows.append({"relation": r["name"], "layer": l, "head": h,
                             "accuracy": float(acc[l, h]), "n_correct": int(corr[r["name"]][l, h]),
                             "n_total": int(tot[r["name"]])})
    print(f"[{tag}] used={used} skipped={skipped}")
    return pd.DataFrame(rows)

train_df = score_dataset(TRAIN_SENTENCES, tag="train")
eval_df = score_dataset(EVAL_SENTENCES, tag="eval")
train_df.to_csv(f"{OUT_DIR}/relation_head_scores_train.csv", index=False)
eval_df.to_csv(f"{OUT_DIR}/relation_head_scores_eval.csv", index=False)
print("saved train/eval score CSVs")

## Candidate heads: rank on train, confirm on eval

In [ ]:
TOP_K = 10
merged = train_df.merge(eval_df, on=["relation", "layer", "head"], suffixes=("_train", "_eval"))
top_rows = []
for r in RELATIONS:
    sub = merged[merged.relation == r["name"]].sort_values("accuracy_train", ascending=False)
    if len(sub) == 0:
        continue
    top_rows.append(sub.iloc[0])
    print(f"\n=== {r['name']}  (train n={int(sub.iloc[0]['n_total_train'])}, eval n={int(sub.iloc[0]['n_total_eval'])}) ===")
    print(sub.head(TOP_K)[["layer", "head", "accuracy_train", "accuracy_eval"]].to_string(index=False))

top_heads_df = pd.DataFrame(top_rows)[["relation", "layer", "head", "accuracy_train", "accuracy_eval"]]
top_heads_df.to_csv(f"{OUT_DIR}/top_relation_heads.csv", index=False)
merged.to_csv(f"{OUT_DIR}/relation_head_scores_merged.csv", index=False)
top_heads_df

## Visualize candidate heads over diffusion time

For each relation's top (train-selected) head, show its attention heatmap across diffusion time on an eval sentence, with the attender row outlined and the `attender -> receiver` cell boxed in red. The fully unmasked frame should show the attender's mass on the receiver.

In [ ]:
def _subset_timesteps(diffusion_steps, n):
    if n >= diffusion_steps:
        return list(range(diffusion_steps))
    return sorted(set(int(i) for i in np.linspace(0, diffusion_steps - 1, n).round().astype(int)))

def save_relation_head_evolution(sentence, layer, head, relation, out_path, title, n_subset=6, ncols=3):
    _, toks, ust, attn_over_time, xt_states = collect_attention_entropy_over_time(
        model=model, tokenizer=tokenizer, text_sequence=sentence,
        layer_idx=layer, head_idx=head, diffusion_steps=DIFFUSION_STEPS, seed=SEED, include_bos=True)
    spans = build_word_spans(toks, include_bos=True)
    aspan = spans[relation["attender_widx"]]; rspan = spans[relation["receiver_widx"]]
    ar = aspan[-1] if ATTENDER_TOKEN == "last" else aspan[0]
    mask_id = int(tokenizer.mask_token_id)
    timesteps = _subset_timesteps(DIFFUSION_STEPS, n_subset)
    nrows = int(np.ceil(len(timesteps) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.2 * nrows))
    axes = np.array(axes).reshape(-1); im = None
    for k, t in enumerate(timesteps):
        ax = axes[k]; A = np.asarray(attn_over_time[t])
        im = ax.imshow(A, aspect="auto", interpolation="nearest")
        labels = [f"{tok} [{'M' if int(v) == mask_id else 'U'}]"
                  for tok, v in zip(toks, xt_states[t][0].tolist())]
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
        ax.axhline(ar, color="red", linewidth=1.0, alpha=0.7)
        for c in rspan:
            ax.add_patch(plt.Rectangle((c - 0.5, ar - 0.5), 1, 1, fill=False, edgecolor="red", linewidth=2.0))
        ax.set_title(f"t = {t}", fontsize=13)
    for k in range(len(timesteps), len(axes)):
        axes[k].axis("off")
    fig.suptitle(title, fontsize=18)
    fig.tight_layout(rect=[0, 0, 0.92, 0.93])  # reserve a strip on the right for the colorbar
    if im is not None:
        cbar_ax = fig.add_axes([0.935, 0.12, 0.015, 0.74])  # [left, bottom, width, height]
        fig.colorbar(im, cax=cbar_ax, label="Attention weight")
    fig.savefig(out_path, format="pdf", bbox_inches="tight"); plt.close(fig)
    return out_path

viz_sentence = next(s for s in EVAL_SENTENCES if valid_template(s))
for _, row in top_heads_df.iterrows():
    rel = next(r for r in RELATIONS if r["name"] == row["relation"])
    L, H = int(row["layer"]), int(row["head"])
    out = f"{OUT_DIR}/evolution_{rel['name']}_L{L}_H{H}.pdf"
    save_relation_head_evolution(
        viz_sentence, L, H, rel, out,
        title=f"{rel['name']}: L{L} H{H} (train={row['accuracy_train']:.2f}, eval={row['accuracy_eval']:.2f}) | {viz_sentence!r}")
    print("saved", out)

## Central result: relation prediction while tokens are masked (held-out)

The table above measures receiver prediction on the *fully unmasked* sentence, which is the least surprising regime. The novel, scientific claim is that relation heads already predict the receiver *before the relation tokens are revealed*. Here we evaluate the top held-out heads (object$\to$verb L4 H1, subject$\to$verb L5 H0) at **every diffusion timestep** on the held-out evaluation set, and report receiver-prediction accuracy when the relation tokens are still **masked** versus **unmasked** (the masked-vs-unmasked gap).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Held-out heads to profile over diffusion time (the central masked-state claim).
CURVE_HEADS = [("object_to_verb", 4, 1), ("subject_to_verb", 5, 0)]
CURVE_EVAL_N = 100            # held-out eval sentences to average over (raise toward 200 for the final run)

MASK_ID = int(tokenizer.mask_token_id)
curve_sentences = [s for s in EVAL_SENTENCES if valid_template(s)][:CURVE_EVAL_N]
print(f"profiling {len(CURVE_HEADS)} heads over {DIFFUSION_STEPS} steps on {len(curve_sentences)} held-out sentences")

def relation_accuracy_over_time(relation_name, layer, head, sentences):
    rel = next(r for r in RELATIONS if r["name"] == relation_name)
    aw, rw = rel["attender_widx"], rel["receiver_widx"]
    S = DIFFUSION_STEPS
    corr = np.zeros(S); tot = np.zeros(S)
    corr_m = np.zeros(S); tot_m = np.zeros(S)   # both relation tokens still masked at step t
    corr_u = np.zeros(S); tot_u = np.zeros(S)   # at least one relation token unmasked
    for sent in sentences:
        _, toks, ust, attn_over_time, xt_states = collect_attention_entropy_over_time(
            model=model, tokenizer=tokenizer, text_sequence=sent,
            layer_idx=layer, head_idx=head, diffusion_steps=S, seed=SEED, include_bos=True)
        spans = build_word_spans(toks, include_bos=True)
        if len(spans) <= max(aw, rw):
            continue
        aspan = spans[aw]; rset = set(spans[rw])
        ar = aspan[-1] if ATTENDER_TOKEN == "last" else aspan[0]
        for t in range(S):
            row = np.asarray(attn_over_time[t])[ar].astype(float).copy()
            if EXCLUDE_BOS: row[0] = -np.inf
            if EXCLUDE_SELF:
                for c in aspan: row[c] = -np.inf
            ok = int(int(np.argmax(row)) in rset)
            corr[t] += ok; tot[t] += 1
            xt = xt_states[t][0].tolist()
            both_masked = (int(xt[ar]) == MASK_ID) and all(int(xt[c]) == MASK_ID for c in rset)
            if both_masked: corr_m[t] += ok; tot_m[t] += 1
            else:           corr_u[t] += ok; tot_u[t] += 1
    return dict(acc=corr/np.maximum(tot,1), acc_masked=corr_m/np.maximum(tot_m,1),
                tot_m=tot_m, tot_u=tot_u, corr_m=corr_m, corr_u=corr_u)

curve_results = {}; summary_rows = []
for name, L, H in CURVE_HEADS:
    r = relation_accuracy_over_time(name, L, H, curve_sentences)
    curve_results[(name, L, H)] = r
    masked_acc = r["corr_m"].sum() / max(r["tot_m"].sum(), 1)
    unmasked_acc = r["corr_u"].sum() / max(r["tot_u"].sum(), 1)
    summary_rows.append({"relation": name, "layer": L, "head": H,
        "acc_fully_masked_t0": float(r["acc"][0]),
        "masked_state_acc": float(masked_acc), "unmasked_acc": float(unmasked_acc),
        "gap": float(unmasked_acc - masked_acc),
        "n_masked": int(r["tot_m"].sum()), "n_unmasked": int(r["tot_u"].sum())})
    print(f"{name} L{L} H{H}: t=0(all masked) acc={r['acc'][0]:.3f} | "
          f"masked-state acc={masked_acc:.3f} (n={int(r['tot_m'].sum())}) | "
          f"unmasked acc={unmasked_acc:.3f} (n={int(r['tot_u'].sum())}) | gap={unmasked_acc-masked_acc:+.3f}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUT_DIR}/relation_accuracy_masked_vs_unmasked.csv", index=False)

# ---- plot: held-out receiver-prediction accuracy vs diffusion timestep (white bg, Type-1) ----
colors = {"object_to_verb": "#1f77b4", "subject_to_verb": "#ff7f0e"}
fig, ax = plt.subplots(figsize=(7.0, 4.6))
xs = range(DIFFUSION_STEPS)
for name, L, H in CURVE_HEADS:
    r = curve_results[(name, L, H)]; c = colors.get(name)
    ax.plot(xs, r["acc"], color=c, linewidth=2.2, label=f"{name.replace('_',' ')} (L{L} H{H})")
    ax.plot(xs, r["acc_masked"], color=c, linewidth=1.4, linestyle="--", alpha=0.7)
ax.axhline(1.0/11, color="gray", linestyle=":", linewidth=1.0, label="chance ($\\approx$1/seq-len)")
ax.set_xlabel("Diffusion step (0 = fully masked, 63 = fully unmasked)")
ax.set_ylabel("Held-out receiver-prediction accuracy")
ax.set_ylim(0, 1.02); ax.grid(alpha=0.25, linestyle="--")
ax.set_title("Relation-head receiver prediction over diffusion time (held-out)\nsolid = all positions, dashed = relation tokens still masked")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/relation_accuracy_over_diffusion_time.pdf", format="pdf", bbox_inches="tight")
plt.show()
summary_df

## Download outputs (Colab)

In [ ]:
import shutil
shutil.make_archive("relation_head_search", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("relation_head_search.zip")
except Exception as exc:
    print("Download helper skipped (local Jupyter? use the file browser):", exc)